# **Fine-Tuning W2V2-Bert for Maltese ASR**

*The following notebook is an adaptation of https://huggingface.co/blog/fine-tune-w2v2-bert*

## Notebook Setup

In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Wed Apr 29 10:28:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.57                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3080        On  |   00000000:07:00.0  On |                  N/A |
|  0%   41C    P8             24W /  370W |    1124MiB /  10240MiB |     35%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Before we start, let's install `datasets` and `transformers`. Also, we need `accelerate` for training, `torchaudio` to load audio files and `jiwer` to evaluate our fine-tuned model using the [word error rate (WER)](https://huggingface.co/metrics/wer) metric ${}^1$.

In [ ]:
# %%capture
# !pip install datasets
# !pip install transformers
# !pip install torchaudio
# !pip install jiwer
# !pip install accelerate -U

In [2]:
import torch

In [3]:
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Using GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"Torch Version: {torch.__version__}")

CUDA Available: True
Using GPU: NVIDIA GeForce RTX 3080
Torch Version: 2.8.0+cu126


## Prepare Data, Tokenizer, Feature Extractor

ASR models transcribe speech to text, which means that we both need a feature extractor that processes the speech signal to the model's input format, *e.g.* a feature vector, and a tokenizer that processes the model's output format to text.

In 🤗 Transformers, the Wav2Vec2-BERT model is thus accompanied by both a tokenizer, called [Wav2Vec2CTCTokenizer](https://huggingface.co/transformers/master/model_doc/wav2vec2.html#wav2vec2ctctokenizer), and a feature extractor, called [SeamlessM4TFeatureExtractor](https://huggingface.co/docs/transformers/v4.36.1/en/model_doc/seamless_m4t#transformers.SeamlessM4TFeatureExtractor) that the model shares with the [first](https://huggingface.co/docs/transformers/main/en/model_doc/seamless_m4t) and [second](https://huggingface.co/docs/transformers/main/en/model_doc/seamless_m4_v2t) versions of Seamless-M4T, as they all process audio in the same way.

Let's start by creating the tokenizer to decode the predicted output classes to the output transcription.

### Create `Wav2Vec2CTCTokenizer`

Remember that Wav2Vec2-like models fine-tuned on CTC transcribe an audio file with a single forward pass by first processing the audio input into a sequence of processed context representations and then using the final vocabulary output layer to classify each context representation to a character that represents the transcription.

The output size of this layer corresponds to the number of tokens in the vocabulary, and therefore only on the labeled dataset used for fine-tuning. So in the first step, we will take a look at the chosen dataset of Common Voice and define a vocabulary based on the transcriptions.

#### Combined Data Loading

In [1]:
import os
import pandas as pd
from pathlib import Path
from datasets import Dataset, concatenate_datasets

# --- Configuration ---
DATA_ROOT = Path.home() / "MalteseProject" / "Data"

def get_audio_path(root_dir, filename, is_nested=False):
    if not filename or filename == "None":
        return None
        
    root_dir = Path(root_dir)
    # Ensure filename has the .wav extension for searching
    search_name = filename if filename.lower().endswith(".wav") else f"{filename}.wav"
    
    if not is_nested:
        direct_path = root_dir / search_name
        return str(direct_path) if direct_path.exists() else None
    
    # Recursive search for MASRI (searching through gender/speaker folders)
    for p in root_dir.rglob(search_name):
        return str(p)
            
    return None

def process_subset(folder_path, source_type, subset_name):
    data_list = []
    folder_path = Path(folder_path)
    
    def safe_str(val):
        return str(val).strip() if pd.notna(val) else ""

    # --- 1 & 2: MASRI / CV (Train & Dev) ---
    if subset_name in ["Train", "Dev"]:
        if source_type == "MASRI":
            csv_name = "masriTrain.csv" if subset_name == "Train" else "masriDev.csv"
            df = pd.read_csv(folder_path / csv_name)
            audio_dir = folder_path / "audio files"
            is_nested = True
            sent_col, id_col = 'transcription_mt', 'filename' # Handle MASRI casing
            # Check if MASRI uses 'Filename' with capital F
            if id_col not in df.columns and 'Filename' in df.columns:
                id_col = 'Filename'
        else: # CV
            csv_name = "cvTrain.csv" if subset_name == "Train" else "cvDev.csv"
            df = pd.read_csv(folder_path / csv_name)
            audio_dir = folder_path / "audio files"
            is_nested = False
            sent_col, id_col = 'sentence_mt', 'filename'

        for _, row in df.iterrows():
            fname = safe_str(row[id_col])
            path = get_audio_path(audio_dir, fname, is_nested=is_nested)
            
            if path:
                data_list.append({
                    "path": path, 
                    "sentence_id": fname,
                    "sentence": safe_str(row[sent_col]), 
                    "sentence_domain": source_type, 
                    "variant": "MT", 
                    "audio": path
                })

    # --- 3: MASRI Test ---
    elif source_type == "MASRI" and subset_name == "Test":
        trans_path = folder_path / "transcription.trans"
        audio_dir = folder_path / "speech"
        
        if trans_path.exists():
            with open(trans_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split(" ", 1)
                    if len(parts) < 2: continue
                    file_id = parts[0]
                    fname = f"{file_id}.wav"
                    path = get_audio_path(audio_dir, fname, is_nested=True)
                    if path:
                        data_list.append({
                            "path": path, "sentence_id": file_id,
                            "sentence": safe_str(parts[1]), 
                            "sentence_domain": "MASRI", "variant": "MT", "audio": path
                        })

    # --- 4: CV Test ---
    elif source_type == "CV" and subset_name == "Test":
        csv_path = folder_path / "FileSequenceCV.csv"
        df = pd.read_csv(csv_path)
        audio_dir = folder_path / "speech"
        corr_col = "If MT-Sentence needs correction, put it here"
        
        for _, row in df.iterrows():
            # Use correction if it exists, else use standard 'sentence'
            final_sent = row[corr_col] if pd.notna(row[corr_col]) and str(row[corr_col]).strip() != "" else row['sentence']
            fname = safe_str(row['path'])
            path = get_audio_path(audio_dir, fname, is_nested=False)
            
            if path:
                data_list.append({
                    "path": path, "sentence_id": fname,
                    "sentence": safe_str(final_sent), 
                    "sentence_domain": "CommonVoice", "variant": "MT", "audio": path
                })

    return Dataset.from_list(data_list)

In [2]:
# --- Process Datasets ---
common_voice_train = concatenate_datasets([
    process_subset(f"{DATA_ROOT}/Train/MASRI Train Release 202501", "MASRI", "Train"),
    process_subset(f"{DATA_ROOT}/Train/CV Train Release 202501", "CV", "Train")
])

common_voice_dev = concatenate_datasets([
    process_subset(f"{DATA_ROOT}/Dev/MASRI Dev Release 202501", "MASRI", "Dev"),
    process_subset(f"{DATA_ROOT}/Dev/CV Dev Release 202501", "CV", "Dev")
])

common_voice_test = concatenate_datasets([
    process_subset(f"{DATA_ROOT}/Test/MASRI", "MASRI", "Test"),
    process_subset(f"{DATA_ROOT}/Test/CV", "CV", "Test")
])

In [3]:
# --- 5. Decode Audio ---
import librosa
import numpy as np

def decode_audio_batched(batch):
    # Process the whole batch in a list comprehension
    audio_data = []
    for path in batch["path"]:
        speech_array, _ = librosa.load(path, sr=16000)
        audio_data.append({
            "path": path,
            "array": np.array(speech_array, dtype=np.float32),
            "sampling_rate": 16000
        })
    batch["audio"] = audio_data
    return batch

# Applying the decoding to all three sets
common_voice_train = common_voice_train.map(decode_audio_batched, batched=True)
common_voice_dev = common_voice_dev.map(decode_audio_batched, batched=True)
common_voice_test = common_voice_test.map(decode_audio_batched, batched=True)

print(f"✅ Loaded and Decoded with Librosa! Train: {len(common_voice_train)} samples.")

Map:   0%|          | 0/8886 [00:00<?, ? examples/s]

Map:   0%|          | 0/1883 [00:00<?, ? examples/s]

Map:   0%|          | 0/1892 [00:00<?, ? examples/s]

✅ Loaded and Decoded with Librosa! Train: 8886 samples.


#### MASRI Only Loading

In [ ]:
import os
import pandas as pd
from pathlib import Path
from datasets import Dataset, concatenate_datasets
import librosa
import numpy as np

# --- Configuration ---
DATA_ROOT = Path.home() / "MalteseProject" / "Data"

def get_audio_path(root_dir, filename, is_nested=False):
    if not filename or filename == "None":
        return None
        
    root_dir = Path(root_dir)
    search_name = filename if filename.lower().endswith(".wav") else f"{filename}.wav"
    
    if not is_nested:
        direct_path = root_dir / search_name
        return str(direct_path) if direct_path.exists() else None
    
    for p in root_dir.rglob(search_name):
        return str(p)
            
    return None

def process_subset(folder_path, source_type, subset_name):
    data_list = []
    folder_path = Path(folder_path)
    
    def safe_str(val):
        return str(val).strip() if pd.notna(val) else ""

    # --- MASRI (Train & Dev) ---
    if subset_name in ["Train", "Dev"] and source_type == "MASRI":
        csv_name = "masriTrain.csv" if subset_name == "Train" else "masriDev.csv"
        df = pd.read_csv(folder_path / csv_name)
        audio_dir = folder_path / "audio files"
        is_nested = True
        sent_col, id_col = 'transcription_mt', 'filename'
        
        if id_col not in df.columns and 'Filename' in df.columns:
            id_col = 'Filename'

        for _, row in df.iterrows():
            fname = safe_str(row[id_col])
            path = get_audio_path(audio_dir, fname, is_nested=is_nested)
            
            if path:
                data_list.append({
                    "path": path, 
                    "sentence_id": fname,
                    "sentence": safe_str(row[sent_col]), 
                    "sentence_domain": source_type, 
                    "variant": "MT", 
                    "audio": path
                })

    # --- MASRI Test ---
    elif source_type == "MASRI" and subset_name == "Test":
        trans_path = folder_path / "transcription.trans"
        audio_dir = folder_path / "speech"
        
        if trans_path.exists():
            with open(trans_path, "r", encoding="utf-8") as f:
                for line in f:
                    parts = line.strip().split(" ", 1)
                    if len(parts) < 2: continue
                    file_id = parts[0]
                    fname = f"{file_id}.wav"
                    path = get_audio_path(audio_dir, fname, is_nested=True)
                    if path:
                        data_list.append({
                            "path": path, "sentence_id": file_id,
                            "sentence": safe_str(parts[1]), 
                            "sentence_domain": "MASRI", "variant": "MT", "audio": path
                        })

    return Dataset.from_list(data_list)

# --- Process Only MASRI Datasets ---
masri_train = process_subset(f"{DATA_ROOT}/Train/MASRI Train Release 202501", "MASRI", "Train")
masri_dev = process_subset(f"{DATA_ROOT}/Dev/MASRI Dev Release 202501", "MASRI", "Dev")
masri_test = process_subset(f"{DATA_ROOT}/Test/MASRI", "MASRI", "Test")

In [2]:
# --- Decode Audio ---
def decode_audio_batched(batch):
    audio_data = []
    for path in batch["path"]:
        speech_array, _ = librosa.load(path, sr=16000)
        audio_data.append({
            "path": path,
            "array": np.array(speech_array, dtype=np.float32),
            "sampling_rate": 16000
        })
    batch["audio"] = audio_data
    return batch

# Applying decoding (Note: names updated to masri_*)
masri_train = masri_train.map(decode_audio_batched, batched=True)
masri_dev = masri_dev.map(decode_audio_batched, batched=True)
masri_test = masri_test.map(decode_audio_batched, batched=True)

print(f"✅ Loaded MASRI! Train: {len(masri_train)}, Dev: {len(masri_dev)}, Test: {len(masri_test)}")

Map:   0%|          | 0/4963 [00:00<?, ? examples/s]

Map:   0%|          | 0/648 [00:00<?, ? examples/s]

Map:   0%|          | 0/668 [00:00<?, ? examples/s]

✅ Loaded MASRI! Train: 4963, Dev: 648, Test: 668


#### Common Voice Only Loading

In [1]:
import os
import pandas as pd
from pathlib import Path
from datasets import Dataset
import librosa
import numpy as np

# --- Configuration ---
DATA_ROOT = Path.home() / "MalteseProject" / "Data"

def get_audio_path(root_dir, filename, is_nested=False):
    if not filename or filename == "None":
        return None
    root_dir = Path(root_dir)
    search_name = filename if filename.lower().endswith(".wav") else f"{filename}.wav"
    
    if not is_nested:
        direct_path = root_dir / search_name
        return str(direct_path) if direct_path.exists() else None
    
    for p in root_dir.rglob(search_name):
        return str(p)
    return None

def process_cv_subset(folder_path, subset_name):
    data_list = []
    folder_path = Path(folder_path)
    
    def safe_str(val):
        return str(val).strip() if pd.notna(val) else ""

    # --- Train & Dev Logic ---
    if subset_name in ["Train", "Dev"]:
        csv_name = "cvTrain.csv" if subset_name == "Train" else "cvDev.csv"
        df = pd.read_csv(folder_path / csv_name)
        audio_dir = folder_path / "audio files"
        
        for _, row in df.iterrows():
            fname = safe_str(row['filename'])
            path = get_audio_path(audio_dir, fname, is_nested=False)
            if path:
                data_list.append({
                    "path": path, 
                    "sentence_id": fname,
                    "sentence": safe_str(row['sentence_mt']), 
                    "sentence_domain": "CommonVoice", 
                    "variant": "MT", 
                    "audio": path
                })

    # --- Test Logic (with Correction Column) ---
    elif subset_name == "Test":
        csv_path = folder_path / "FileSequenceCV.csv"
        df = pd.read_csv(csv_path)
        audio_dir = folder_path / "speech"
        corr_col = "If MT-Sentence needs correction, put it here"
        
        for _, row in df.iterrows():
            # logic: Use correction if it exists, otherwise use standard 'sentence'
            final_sent = row[corr_col] if pd.notna(row[corr_col]) and str(row[corr_col]).strip() != "" else row['sentence']
            fname = safe_str(row['path'])
            path = get_audio_path(audio_dir, fname, is_nested=False)
            
            if path:
                data_list.append({
                    "path": path, 
                    "sentence_id": fname,
                    "sentence": safe_str(final_sent), 
                    "sentence_domain": "CommonVoice", 
                    "variant": "MT", 
                    "audio": path
                })

    return Dataset.from_list(data_list)

# --- 1. Load the Splits Individually ---
# These are now kept as separate Dataset objects
common_voice_train = process_cv_subset(DATA_ROOT / "Train" / "CV Train Release 202501", "Train")
common_voice_dev = process_cv_subset(DATA_ROOT / "Dev" / "CV Dev Release 202501", "Dev")
common_voice_test = process_cv_subset(DATA_ROOT / "Test" / "CV", "Test")

# --- Verification ---
print(f"Train size: {len(common_voice_train)}")
print(f"Dev size: {len(common_voice_dev)}")
print(f"Test size: {len(common_voice_test)}")

Train size: 3923
Dev size: 1235
Test size: 1224


In [2]:
# --- 3. Decode Audio ---
def decode_audio(batch):
    # Note: Using librosa to match your target format
    speech_array, _ = librosa.load(batch["path"], sr=16000)
    batch["audio"] = {
        "path": batch["path"],
        "array": np.array(speech_array, dtype=np.float32),
        "sampling_rate": 16000
    }
    return batch

# Apply decoding (this processes one row at a time as per your old script)
common_voice_train = common_voice_train.map(decode_audio)
common_voice_test = common_voice_test.map(decode_audio)

print(f"✅ Loaded CV! Train+Dev: {len(common_voice_train)}, Test: {len(common_voice_test)}")

Map:   0%|          | 0/3923 [00:00<?, ? examples/s]

Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

✅ Loaded CV! Train+Dev: 3923, Test: 1224


#### Utilities

In [6]:
def print_dataset_stats(dataset, name):
    total_utterances = len(dataset)
    # Extract the 'sentence' column and find unique entries
    unique_utterances = len(set(dataset["sentence"]))
    
    print(f"--- Statistics for {name} ---")
    print(f"Total Utterances:  {total_utterances}")
    print(f"Unique Sentences:  {unique_utterances}")
    if total_utterances > 0:
        print(f"Redundancy Ratio: {((total_utterances - unique_utterances) / total_utterances):.2%}")
    print("-" * 30)

# Usage
print_dataset_stats(common_voice_train, "Common Voice Train")
print_dataset_stats(common_voice_dev, "Common Voice Dev")
print_dataset_stats(common_voice_test, "Common Voice Test")

--- Statistics for Common Voice Train ---
Total Utterances:  3923
Unique Sentences:  3499
Redundancy Ratio: 10.81%
------------------------------
--- Statistics for Common Voice Dev ---
Total Utterances:  1235
Unique Sentences:  1195
Redundancy Ratio: 3.24%
------------------------------
--- Statistics for Common Voice Test ---
Total Utterances:  1224
Unique Sentences:  1176
Redundancy Ratio: 3.92%
------------------------------


In [ ]:
import librosa

def calculate_duration(dataset):
    # Sum up durations of all paths in the 'audio' column
    total_seconds = sum(librosa.get_duration(path=x) for x in dataset['audio'])
    
    total_hours = total_seconds / 3600
    print(f"Total Duration: {total_seconds:.2f} seconds")
    print(f"Total Duration: {total_hours:.2f} hours")
    
    return total_hours


calculate_duration(common_voice_test)

Total Duration: 5657.74 seconds
Total Duration: 1.57 hours


1.5715933333333323

In [2]:
def add_duration(example):
    # librosa.get_duration reads the header without loading the full waveform
    example["duration"] = librosa.get_duration(path=example["path"])
    return example

# Apply to your datasets
cv_train_with_duration = common_voice_train.map(add_duration)

# Calculate Average
avg_duration = np.mean(cv_train_with_duration["duration"])
print(f"Average Duration: {avg_duration:.2f} seconds")
print(f"Total Hours: {sum(cv_train_with_duration['duration']) / 3600:.2f}")

Map:   0%|          | 0/3923 [00:00<?, ? examples/s]

Average Duration: 4.76 seconds
Total Hours: 5.19


### Textual Preprocessing

Many ASR datasets only provide the target text, `'sentence'` for each audio array `'audio'` and file `'path'`. Common Voice actually provides much more information about each audio file, such as the `'accent'`, etc. Keeping the notebook as general as possible, we only consider the transcribed text for fine-tuning.

In [4]:
print(f"Current columns: {common_voice_train.column_names}")

Current columns: ['path', 'sentence_id', 'sentence', 'sentence_domain', 'variant', 'audio']


In [ ]:
common_voice_train = common_voice_train.remove_columns(["accents", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])
common_voice_test = common_voice_test.remove_columns(["accents", "age", "client_id", "down_votes", "gender", "locale", "segment", "up_votes"])

Let's write a short function to display some random samples of the dataset and run it a couple of times to get a feeling for the transcriptions.

In [3]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

In [4]:
show_random_elements(common_voice_train.remove_columns(["path", "audio"]), num_examples=10)

,sentence_id,sentence,sentence_domain,variant
0,common_voice_mt_21341894.wav,"Fil-fatt Maradona kien ukoll ma' Barcelona, għoxrin sena qabel.",CommonVoice,MT
1,common_voice_mt_21335861.wav,Caruana ntlaqat minn sitt tiri.,CommonVoice,MT
2,common_voice_mt_21279043.wav,Ilu jżomm mal-Inter sa mit-tfulija tiegħu.,CommonVoice,MT
3,common_voice_mt_21278871.wav,F'dan l-istadju trid issir proposta formali biex wieħed ikun jista' janalizzaha.,CommonVoice,MT
4,common_voice_mt_21171087.wav,Il-Ministru għat-Trasport u l-Infrastruttura jekk jogħġbu jressaq ir-Riżoluzzjonijiet tiegħu.,CommonVoice,MT
5,common_voice_mt_21330306.wav,U x'mhux hekk,CommonVoice,MT
6,common_voice_mt_22764241.wav,Ma naqsux iċ-ċelebrazzjonijiet tat-tim flimkien mal-partitarji numerużi ta' Balzan.,CommonVoice,MT
7,common_voice_mt_21240970.wav,"Madankollu, ir-riċerka tiswa l-flus.",CommonVoice,MT
8,common_voice_mt_21506449.wav,Bihom huma jgħinu lill-missjunarji Maltin tal-pajjiżi foqra tat-tielet dinja.,CommonVoice,MT
9,common_voice_mt_21241357.wav,Imma bħala Nisranija kapaċi nara l-affarijiet min-naħa l-oħra wkoll.,CommonVoice,MT


Alright! The transcriptions look fairly clean. Having translated the transcribed sentences, it seems that the language corresponds more to written-out text than noisy dialogue. This makes sense considering that [Common Voice](https://huggingface.co/datasets/mozilla-foundation/common_voice_16_0) is a crowd-sourced read speech corpus.

We can see that the transcriptions contain some special characters, such as `,.?!;:`. Without a language model, it is much harder to classify speech chunks to such special characters because they don't really correspond to a characteristic sound unit. *E.g.*, the letter `"s"` has a more or less clear sound, whereas the special character `"."` does not.
Also in order to understand the meaning of a speech signal, it is usually not necessary to include special characters in the transcription.

Let's simply remove all characters that don't contribute to the meaning of a word and cannot really be represented by an acoustic sound and normalize the text.

In [5]:
import re
chars_to_remove_regex = '[\,\?\.\!\;\:\"\“\%\‘\”\�\»\«]'

def remove_special_characters(batch):
    # remove special characters
    batch["sentence"] = re.sub(chars_to_remove_regex, '', batch["sentence"]).lower()

    return batch

In [6]:
common_voice_train = common_voice_train.map(remove_special_characters)
common_voice_test = common_voice_test.map(remove_special_characters)
common_voice_dev = common_voice_dev.map(remove_special_characters)

Map:   0%|          | 0/3923 [00:00<?, ? examples/s]

Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

Map:   0%|          | 0/1235 [00:00<?, ? examples/s]

Let's look at the processed text labels again.

In [19]:
show_random_elements(common_voice_test.remove_columns(["path","audio"]))

,sentence_id,sentence,sentence_domain,variant
0,common_voice_mt_21194368.wav,allura nbidlet,CommonVoice,MT
1,common_voice_mt_21481333.wav,it-talbiet li ġew milqugħa kif huma maqsumin,CommonVoice,MT
2,common_voice_mt_21274909.wav,issa ġie ssuġġerit biex ir-regolatur ma tkunx persuna waħda,CommonVoice,MT
3,common_voice_mt_21398060.wav,qed nifhem il-punt,CommonVoice,MT
4,common_voice_mt_21431108.wav,żied jgħid li l-kunsill lokali ħela tliet snin fi ġlied intern,CommonVoice,MT
5,common_voice_mt_21553965.wav,f'dan is-sit ukoll saru bjar apposta biex jinġabru dawn il-gassijiet,CommonVoice,MT
6,common_voice_mt_21397684.wav,kif beda l-interess tiegħek fl-atletika,CommonVoice,MT
7,common_voice_mt_21320520.wav,qablek min kien hemm,CommonVoice,MT
8,common_voice_mt_21240811.wav,it-tim ta' arsenal huwa tim verament qawwi u bi storja kbira,CommonVoice,MT
9,common_voice_mt_21370110.wav,newsbook dot com dot mt hija waħda minnhom,CommonVoice,MT


In CTC, it is common to classify speech chunks into letters, so we will do the same here.
Let's extract all distinct letters of the training and test data and build our vocabulary from this set of letters.

We write a mapping function that concatenates all transcriptions into one long transcription and then transforms the string into a set of chars.
It is important to pass the argument `batched=True` to the `map(...)` function so that the mapping function has access to all transcriptions at once.

In [8]:
def extract_all_chars(batch):
  all_text = " ".join(batch["sentence"])
  vocab = list(set(all_text))
  return {"vocab": [vocab], "all_text": [all_text]}

In [9]:
vocab_train = common_voice_train.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_train.column_names)
vocab_test = common_voice_test.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_test.column_names)
vocab_dev = common_voice_dev.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_dev.column_names)

Map:   0%|          | 0/3923 [00:00<?, ? examples/s]

Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

Map:   0%|          | 0/1235 [00:00<?, ? examples/s]

Now, we create the union of all distinct letters in the training dataset and test dataset and convert the resulting list into an enumerated dictionary.

In [11]:
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]) | set(vocab_dev["vocab"][0]))

In [12]:
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
vocab_dict

{' ': 0,
 "'": 1,
 '-': 2,
 '`': 3,
 'a': 4,
 'b': 5,
 'c': 6,
 'd': 7,
 'e': 8,
 'f': 9,
 'g': 10,
 'h': 11,
 'i': 12,
 'j': 13,
 'k': 14,
 'l': 15,
 'm': 16,
 'n': 17,
 'o': 18,
 'p': 19,
 'q': 20,
 'r': 21,
 's': 22,
 't': 23,
 'u': 24,
 'v': 25,
 'w': 26,
 'x': 27,
 'y': 28,
 'z': 29,
 'à': 30,
 'á': 31,
 'è': 32,
 'é': 33,
 'ì': 34,
 'ò': 35,
 'ù': 36,
 'ć': 37,
 'ċ': 38,
 'ġ': 39,
 'ħ': 40,
 'ż': 41}

Cleaning up a dataset is a back-and-forth process that needs to be done with care.

Looking at the separate letters in the training and test datasets, and after discussing them with a native speaker of the target language (thanks [Mishig](https://github.com/mishig25)  for taking a look), we can see Latin characters that we are going to remove for two reasons:
1. the CTC algorithm benefits from reduced vocabulary size
2. in this example, we are concentrating entirely on the Mongolian alphabet.

In [13]:
import sys
import re
from pathlib import Path

masri_root = str(Path(".").resolve() / "MASRI")

if masri_root not in sys.path:
    sys.path.append(masri_root)

# Importing tokeniser
try:
    from masri.tokenise.tokenise import MTWordTokenizer
    tokenizer = MTWordTokenizer()
    print("MASRI Tokenizer loaded successfully!")
except ImportError as e:
    print(f"Failed to load MASRI: {e}")


# 3. Define the cleaning function
# Removing digits and unecessary characters that show up in the datasets
chars_to_remove_regex = r'[\d\_\`]' #Removed ` because only appears twice.

def process_maltese_text(batch):
    # Initial cleaning (remove unwanted symbols/punctuation)
    clean_text = re.sub(chars_to_remove_regex, '', batch["sentence"]).lower()

    # Define your Normalization Mapping
    # This maps rare accented characters to their base versions
    mapping = {
         'á': 'a',
         'é': 'e',
         'ć': 'ċ'  

     }

     # 3. Apply the mapping
    for char, replacement in mapping.items():
         clean_text = clean_text.replace(char, replacement)

    # 4. Use MASRI tokenizer to handle Maltese specifics (clitics, etc.)
    # Note: Ensure the tokenizer is initialized outside this function
    #tokens = tokenizer.tokenize(clean_text)
    batch["sentence"] = clean_text.strip()
    
    # 5. Re-join into a clean string for ASR training
    #batch["sentence"] = " ".join(tokens)
    return batch

MASRI Tokenizer loaded successfully!


In [ ]:
# 4. Apply to your datasets
masri_train = masri_train.map(process_maltese_text)
masri_test = masri_test.map(process_maltese_text)
masri_dev = masri_dev.map(process_maltese_text)

In [15]:
# 4. Apply to your datasets
common_voice_train = common_voice_train.map(process_maltese_text)
common_voice_test = common_voice_test.map(process_maltese_text)
common_voice_dev = common_voice_dev.map(process_maltese_text)

Map:   0%|          | 0/3923 [00:00<?, ? examples/s]

Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

Map:   0%|          | 0/1235 [00:00<?, ? examples/s]

In [16]:
vocab_train = common_voice_train.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_train.column_names)
vocab_test = common_voice_test.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_test.column_names)
vocab_dev = common_voice_dev.map(extract_all_chars, batched=True, batch_size=-1, keep_in_memory=True, remove_columns=common_voice_dev.column_names)

Map:   0%|          | 0/3923 [00:00<?, ? examples/s]

Map:   0%|          | 0/1224 [00:00<?, ? examples/s]

Map:   0%|          | 0/1235 [00:00<?, ? examples/s]

In [17]:
vocab_list = list(set(vocab_train["vocab"][0]) | set(vocab_test["vocab"][0]) | set(vocab_dev["vocab"][0]))

In [18]:
vocab_dict = {v: k for k, v in enumerate(sorted(vocab_list))}
vocab_dict

{' ': 0,
 "'": 1,
 '-': 2,
 'a': 3,
 'b': 4,
 'c': 5,
 'd': 6,
 'e': 7,
 'f': 8,
 'g': 9,
 'h': 10,
 'i': 11,
 'j': 12,
 'k': 13,
 'l': 14,
 'm': 15,
 'n': 16,
 'o': 17,
 'p': 18,
 'q': 19,
 'r': 20,
 's': 21,
 't': 22,
 'u': 23,
 'v': 24,
 'w': 25,
 'x': 26,
 'y': 27,
 'z': 28,
 'à': 29,
 'è': 30,
 'ì': 31,
 'ò': 32,
 'ù': 33,
 'ċ': 34,
 'ġ': 35,
 'ħ': 36,
 'ż': 37}

Cool, we see that all letters of the alphabet occur in the dataset (which is not really surprising) and we also extracted the special character `" "`. Note that we did not exclude this special character because:

The model has to learn to predict when a word is finished or else the model prediction would always be a sequence of chars which would make it impossible to separate words from each other.

One should always keep in mind that pre-processing is a very important step before training your model. E.g., we don't want our model to differentiate between `a` and `A` just because we forgot to normalize the data. The difference between `a` and `A` does not depend on the "sound" of the letter at all, but more on grammatical rules - *e.g.* use a capitalized letter at the beginning of the sentence. So it is sensible to remove the difference between capitalized and non-capitalized letters so that the model has an easier time learning to transcribe speech.




To make it clearer that `" "` has its own token class, we give it a more visible character `|`. In addition, we also add an "unknown" token so that the model can later deal with characters not encountered in Common Voice's training set.

In [27]:
vocab_dict["|"] = vocab_dict[" "]
del vocab_dict[" "]

Finally, we also add a padding token that corresponds to CTC's "*blank token*".


The "blank token" is a core component of the CTC algorithm. For more information, please take a look at the "Alignment" section [here](https://distill.pub/2017/ctc/).

In [28]:
vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)
len(vocab_dict)

36

Cool, now our vocabulary is complete and consists of 37 tokens, which means that the linear layer that we will add on top of the pretrained W2V-BERT checkpoint will have an output dimension of 37.

Let's now save the vocabulary as a json file.

In [36]:
import json
with open('vocab.json', 'w') as vocab_file:
    json.dump(vocab_dict, vocab_file)

In a final step, we use the json file to load the vocabulary into an instance of the `Wav2Vec2CTCTokenizer` class.

In [37]:
from transformers import Wav2Vec2CTCTokenizer

tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("./", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|")

### Create `SeamlessM4TFeatureExtractor`

The role of the `SeamlessM4TFeatureExtractor` is to prepare the raw audio input in a format that the model can "understand". It therefore maps the sequence of one-dimensional amplitude values (aka the raw audio input) to a two-dimensional matrix of log-mel spectrogram values. The latter encodes the signal frequency information as a function of time. See [this section](https://huggingface.co/learn/audio-course/chapter1/audio_data#the-frequency-spectrum) from the Audio Transformers course to learn more about spectrograms and why they are important.

Unlike the tokenizer, the feature extractor doesn't need to be "learned" from the data, so we can load it directly from the [initial model checkpoint](https://huggingface.co/facebook/w2v-bert-2.0).


In [21]:
from transformers import SeamlessM4TFeatureExtractor

feature_extractor = SeamlessM4TFeatureExtractor(feature_size=80, num_mel_bins=80, sampling_rate=16000, padding_value=0.0)

Great, W2V-BERT's feature extraction pipeline is thereby fully defined!

For improved user-friendliness, the feature extractor and tokenizer are *wrapped* into a single `Wav2Vec2BertProcessor` class so that one only needs a `model` and `processor` object.

In [22]:
from transformers import Wav2Vec2BertProcessor

processor = Wav2Vec2BertProcessor(feature_extractor=feature_extractor, tokenizer=tokenizer)

Next, we can prepare the dataset.

### Preprocess Data

So far, we have not looked at the actual values of the speech signal but just the transcription. In addition to `sentence`, our datasets include two more column names `path` and `audio`. `path` states the absolute path of the audio file. Let's take a look.


In [ ]:
# !pip install torchcodec

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.2 MB 4.2 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 10.4 MB/s  0:00:00


In [28]:
common_voice_train[0]["path"]

'/home/ian/MalteseProject/Data/Train/MASRI Train Release 202501/audio files/female/F_01/MSRHS_F_01_P02U001_0001.wav'

W2V-BERT expects the input in the format of a 1-dimensional array of 16 kHz. This means that the audio file has to be loaded and resampled.

 Thankfully, `datasets` does this automatically by calling the other column `audio`. Let try it out.

In [45]:
common_voice_train[0]["audio"]

{'array': [0.005828857421875,
  0.008819580078125,
  0.006988525390625,
  0.0079345703125,
  0.00732421875,
  0.007232666015625,
  0.007720947265625,
  0.00823974609375,
  0.00909423828125,
  0.0087890625,
  0.008087158203125,
  0.008056640625,
  0.007354736328125,
  0.008270263671875,
  0.007354736328125,
  0.00726318359375,
  0.0076904296875,
  0.007781982421875,
  0.00775146484375,
  0.0067138671875,
  0.0074462890625,
  0.008056640625,
  0.006805419921875,
  0.006378173828125,
  0.00738525390625,
  0.0069580078125,
  0.007537841796875,
  0.00689697265625,
  0.005828857421875,
  0.00543212890625,
  0.007080078125,
  0.006988525390625,
  0.006072998046875,
  0.006317138671875,
  0.005828857421875,
  0.006439208984375,
  0.006072998046875,
  0.005462646484375,
  0.0057373046875,
  0.00689697265625,
  0.005645751953125,
  0.005126953125,
  0.005615234375,
  0.005706787109375,
  0.00634765625,
  0.004180908203125,
  0.005126953125,
  0.005340576171875,
  0.0047607421875,
  0.00527954101


Great, we can see that the audio file has automatically been loaded. This is thanks to the new [`"Audio"` feature](https://huggingface.co/docs/datasets/package_reference/main_classes.html?highlight=audio#datasets.Audio) introduced in `datasets == 4.13.3`, which loads and resamples audio files on-the-fly upon calling.

In the example above we can see that the audio data is loaded with a sampling rate of 48kHz whereas Wav2Vec2-BERT was pre-trained at a sampling rate of 16kHz. The sampling rate plays an important role in that it defines how many data points of the speech signal are measured per second. Therefore, sampling with a higher sampling rate results in a better approximation of the *real* speech signal but also necessitates more values per second.

A pre-trained checkpoint expects its input data to have been sampled more or less from the same distribution as the data it was trained on. The same speech signals sampled at two different rates have a very different distribution, *e.g.*, doubling the sampling rate results in data points being twice as long. Thus,
before fine-tuning a pre-trained checkpoint of an ASR model, it is crucial to verify that the sampling rate of the data that was used to pre-train the model matches the sampling rate of the dataset used to fine-tune the model.

Let's listen to a couple of audio files to better understand the dataset and verify that the audio was correctly loaded.

**Note**: *You can click the following cell a couple of times to listen to different speech samples.*

In [30]:
import IPython.display as ipd
import numpy as np
import random

rand_int = random.randint(0, len(common_voice_train)-1)

print(common_voice_train[rand_int]["sentence"])
ipd.Audio(data=common_voice_train[rand_int]["audio"]["array"], autoplay=True, rate=16000)

jinħabbu bil- bosta għarajjes flimkien u mat- tabib callus ma tantx tista' tiċċajta


It seems like the data is now correctly loaded and resampled.

It can be heard, that the speakers change along with their speaking rate, accent, and background environment, etc. Overall, the recordings sound acceptably clear though, which is to be expected from a crowd-sourced read speech corpus.

Let's do a final check that the data is correctly prepared, by printing the shape of the speech input, its transcription, and the corresponding sampling rate.

**Note**: *You can click the following cell a couple of times to verify multiple samples.*

In [50]:
import random
import numpy as np

rand_int = random.randint(0, len(common_voice_train)-1)

# Get the array data
audio_data = common_voice_train[rand_int]["audio"]["array"]

print("Target text:", common_voice_train[rand_int]["sentence"])
# We wrap it in np.array here just to check the shape
print("Input array shape:", np.array(audio_data).shape) 
print("Sampling rate:", common_voice_train[rand_int]["audio"]["sampling_rate"])

Target text: il- fieragħ tal- fizzjal imħawwad b' dak kollu li kien ġralu wara li ħallieha laqqas kien jiftakar iżjed fil- laqgħa li kellu qawl il- lejl ma' marjetta
Input array shape: (128521,)
Sampling rate: 16000


Good! Everything looks fine - the data is a 1-dimensional array, the sampling rate always corresponds to 16kHz, and the target text is normalized.

Finally, we can leverage `Wav2Vec2BertProcessor` to process the data to the format expected by `Wav2Vec2BertForCTC` for training. To do so let's make use of Dataset's [`map(...)`](https://huggingface.co/docs/datasets/package_reference/main_classes.html?highlight=map#datasets.DatasetDict.map) function.

First, we load and resample the audio data, simply by calling `batch["audio"]`.
Second, we extract the `input_features` from the loaded audio file. In our case, the `Wav2Vec2BertProcessor` creates a more complex representation as the raw waveform, known as [Log-Mel feature extraction](https://en.wikipedia.org/wiki/Mel-frequency_cepstrum).
Third, we encode the transcriptions to label ids.


In [31]:
def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor(audio["array"], sampling_rate=audio["sampling_rate"]).input_features[0]
    batch["input_length"] = len(batch["input_features"])

    batch["labels"] = processor(text=batch["sentence"]).input_ids
    return batch

Let's apply the data preparation function to all examples.

In [32]:
common_voice_train = common_voice_train.map(prepare_dataset, remove_columns=common_voice_train.column_names)
common_voice_test = common_voice_test.map(prepare_dataset, remove_columns=common_voice_test.column_names)

Map:   0%|          | 0/8886 [00:00<?, ? examples/s]

Map:   0%|          | 0/1892 [00:00<?, ? examples/s]

**Note**: `datasets` automatically takes care of audio loading and resampling. If you wish to implement your own costumized data loading/sampling, feel free to just make use of the `"path"` column instead and disregard the `"audio"` column.

Awesome, now we are ready to start training!

## Training

The data is processed so that we are ready to start setting up the training pipeline. We will make use of 🤗's [Trainer](https://huggingface.co/transformers/master/main_classes/trainer.html?highlight=trainer) for which we essentially need to do the following:

- Define a data collator. In contrast to most NLP models, W2V-BERT has a much larger input length than output length. Given the large input sizes, it is much more efficient to pad the training batches dynamically meaning that all training samples should only be padded to the longest sample in their batch and not the overall longest sample. Therefore, fine-tuning W2V-BERT requires a special padding data collator, which we will define below.

- Evaluation metric. During training, the model should be evaluated on the word error rate. We should define a `compute_metrics` function accordingly

- Load a pretrained checkpoint. We need to load a pretrained checkpoint and configure it correctly for training.

- Define the training configuration.

After having fine-tuned the model, we will correctly evaluate it on the test data and verify that it has indeed learned to correctly transcribe speech.

### Set-up Trainer

Let's start by defining the data collator. The code for the data collator was copied from [this example](https://github.com/huggingface/transformers/blob/7e61d56a45c19284cfda0cee8995fb552f6b1f4e/examples/pytorch/speech-recognition/run_speech_recognition_ctc.py#L219).

Without going into too many details, in contrast to the common data collators, this data collator treats the `input_features` and `labels` differently and thus applies to separate padding functions on them. This is necessary because in speech input and output are of different modalities meaning that they should not be treated by the same padding function.
Analogous to the common data collators, the padding tokens in the labels with `-100` so that those tokens are **not** taken into account when computing the loss.

In [33]:
import torch

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Union

@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2BertProcessor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lenghts and need
        # different padding methods
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt",
        )

        labels_batch = self.processor.pad(
            labels=label_features,
            padding=self.padding,
            return_tensors="pt",
        )
        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        batch["labels"] = labels

        return batch

In [34]:
data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

Next, the evaluation metric is defined. As mentioned earlier, the
predominant metric in ASR is the word error rate (WER), hence we will use it in this notebook as well.

In [35]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

The model will return a sequence of logit vectors:
$\mathbf{y}_1, \ldots, \mathbf{y}_m$ with $\mathbf{y}_1 = f_{\theta}(x_1, \ldots, x_n)[0]$ and $n >> m$.

A logit vector $\mathbf{y}_1$ contains the log-odds for each word in the vocabulary we defined earlier, thus $\text{len}(\mathbf{y}_i) =$ `config.vocab_size`. We are interested in the most likely prediction of the model and thus take the `argmax(...)` of the logits. Also, we transform the encoded labels back to the original string by replacing `-100` with the `pad_token_id` and decoding the ids while making sure that consecutive tokens are **not** grouped to the same token in CTC style ${}^1$.

In [36]:
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    # Handle the -100 labels
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    # Compute both metrics
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer, "cer": cer}

Now, we can load the pretrained checkpoint of [Wav2Vec2-XLS-R-300M](https://huggingface.co/facebook/wav2vec2-xls-r-300m). The tokenizer's `pad_token_id` must be to define the model's `pad_token_id` or in the case of `Wav2Vec2BertForCTC` also CTC's *blank token* ${}^2$. To save GPU memory, we enable PyTorch's [gradient checkpointing](https://pytorch.org/docs/stable/checkpoint.html) and also set the loss reduction to "*mean*".

Since, we're only training a small subset of weights, the model is not prone to overfitting. Therefore, we make sure to disable all dropout layers.

**Note**: When using this notebook to train W2V-BERT on another language of Common Voice those hyper-parameter settings might not work very well. Feel free to adapt those depending on your use case.

In [37]:
from transformers import Wav2Vec2BertForCTC

model = Wav2Vec2BertForCTC.from_pretrained(
    "facebook/w2v-bert-2.0",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.0,
    layerdrop=0.0,
    ctc_loss_reduction="mean",
    ctc_zero_infinity=True, #ADDED BY IAN TO TRY TO PREVENT NAN
    add_adapter=True,
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

Some weights of Wav2Vec2BertForCTC were not initialized from the model checkpoint at facebook/w2v-bert-2.0 and are newly initialized: ['lm_head.bias', 'lm_head.weight', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.intermediate_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.bias', 'wav2vec2_bert.adapter.layers.0.ffn.output_dense.weight', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.ffn_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.residual_conv.bias', 'wav2vec2_bert.adapter.layers.0.residual_conv.weight', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.bias', 'wav2vec2_bert.adapter.layers.0.residual_layer_norm.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_k.weight', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.bias', 'wav2vec2_bert.adapter.layers.0.self_attn.linear_out.weight', 'wav2vec2_ber

In a final step, we define all parameters related to training.
To give more explanation on some of the parameters:
- `group_by_length` makes training more efficient by grouping training samples of similar input length into one batch. This can significantly speed up training time by heavily reducing the overall number of useless padding tokens that are passed through the model
- `learning_rate` was heuristically tuned until fine-tuning has become stable. Note that those parameters strongly depend on the Common Voice dataset and might be suboptimal for other speech datasets.

For more explanations on other parameters, one can take a look at the [docs](https://huggingface.co/transformers/master/main_classes/trainer.html?highlight=trainer#trainingarguments).

During training, a checkpoint will be uploaded asynchronously to the hub every 600 training steps. It allows you to also play around with the demo widget even while your model is still training.

**Note**: If one does not want to upload the model checkpoints to the hub, simply set `push_to_hub=False`.

In [39]:
from transformers import TrainingArguments

training_args = TrainingArguments(
  output_dir=repo_name,
  group_by_length=True,
  per_device_train_batch_size=16,
  gradient_accumulation_steps=2,
  evaluation_strategy="steps",
  num_train_epochs=10,
  gradient_checkpointing=True,
  fp16=True,
  save_steps=600,
  eval_steps=300,
  logging_steps=300,
  learning_rate=5e-5,
  warmup_steps=500,
  save_total_limit=2,
  push_to_hub=True,
)

NameError: name 'repo_name' is not defined

In [38]:
from transformers import TrainingArguments

# Output path
local_output_dir = "/mnt/c/Users/Ian/Documents/TeziTesting/Runs/runTokenised"

training_args = TrainingArguments(
  output_dir=local_output_dir,
  group_by_length=True,
  
  # --- GPU VRAM ADJUSTMENTS (Safe for 3080) ---
  per_device_train_batch_size=4,
  gradient_accumulation_steps=8,
  gradient_checkpointing=True,
  fp16=True,
  bf16=False, 


  # Prevents NaN by capping giant gradient spikes
  max_grad_norm=1.0,
  
  # --- TRAINING SETTINGS ---
  eval_strategy="steps",
  save_strategy="steps",
  num_train_epochs=10,
  # learning_rate=2e-5,
  learning_rate=5e-5,  #Changed this since Im resuming training
  warmup_steps=500,
  
  # --- LOGGING & LOCAL SAVING ---
  save_steps=300,       # Saves a checkpoint folder every 300 steps
  eval_steps=300,       # Tests the model every 300 steps
  logging_steps=100,    # Shows you the loss/progress every 100 steps
  save_total_limit=3,   # Keeps only the 2 most recent checkpoints (saves disk space)

  # Best model logic
  load_best_model_at_end=True, # Automatically loads the best model when training finishes
  metric_for_best_model="wer", # Decide 'best' based on Word Error Rate
  greater_is_better=False,
  
  # --- DISABLE HUB/GIT ---
  push_to_hub=False,            # No Hugging Face upload
  report_to="none",             # Prevents trying to log to online tools like WandB
)

Now, all instances can be passed to Trainer and we are ready to start training!

In [39]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=common_voice_train,
    eval_dataset=common_voice_test,
    # Use processing_class instead of tokenizer
    processing_class=processor, 
)



---

${}^1$ To allow models to become independent of the speaker rate, in CTC, consecutive tokens that are identical are simply grouped as a single token. However, the encoded labels should not be grouped when decoding since they don't correspond to the predicted tokens of the model, which is why the `group_tokens=False` parameter has to be passed. If we wouldn't pass this parameter a word like `"hello"` would incorrectly be encoded, and decoded as `"helo"`.

${}^2$ The blank token allows the model to predict a word, such as `"hello"` by forcing it to insert the blank token between the two l's. A CTC-conform prediction of `"hello"` of our model would be `[PAD] [PAD] "h" "e" "e" "l" "l" [PAD] "l" "o" "o" [PAD]`.

### Training

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 44, 'bos_token_id': 43}.


Step,Training Loss,Validation Loss


## KenLM Experiments

In [17]:
import torch
import librosa
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from jiwer import wer
from tqdm import tqdm

# 1. Setup paths
base_path = Path(".") 
my_model_path = base_path / "Runs" / "checkpoint-5690"
LM_PATH = str(base_path / "3_untok_red.bin")

device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load Model and Processor
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

# 3. Prepare Decoder Vocab
vocab = processor.tokenizer.get_vocab()
sorted_vocab = [k for k, v in sorted(vocab.items(), key=lambda item: item[1])]


ARPA_PATH = str(base_path / "3_untok_red.arpa") # Path to  ARPA file

# 1. Extract Unigrams 
def get_unigram_list(arpa_path):
    unigrams = []
    with open(arpa_path, "r") as f:
        is_unigram_section = False
        for line in f:
            if "\\1-grams:" in line:
                is_unigram_section = True
                continue
            if "\\2-grams:" in line or line.startswith("\\end\\"):
                break
            if is_unigram_section:
                parts = line.strip().split()
                if len(parts) >= 2:
                    word = parts[1]
                    if word not in ["<s>", "</s>", "<unk>"]:
                        unigrams.append(word)
    return unigrams

print("Extracting unigrams...")
UNIGRAM_LIST = get_unigram_list(ARPA_PATH)

def optimize_with_dataset(dataset):
    all_logits_numpy = []
    all_ground_truths = []
    greedy_preds = []

    print(f"--- Step 1: Generating Logits for {len(dataset)} samples ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        # Use the logic from your attempt: Load from the 'audio' dictionary in the dataset
        audio = batch["audio"]["array"] 
        
        # 4. Convert audio to format the model expects (Your Step 5 logic)
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        # 5. Run Inference (Your Step 6 logic)
        with torch.no_grad():
            logits = model(input_features).logits
        
        # Store for KenLM
        all_logits_numpy.append(logits.cpu().numpy()[0])
        
        # Store Ground Truth
        all_ground_truths.append(batch["sentence"].lower())
        
        # Store Greedy (Your Step 7 logic)
        pred_ids = torch.argmax(logits, dim=-1)
        greedy_preds.append(processor.batch_decode(pred_ids)[0].lower())

    # Calculate Baseline WER
    baseline_wer = wer(all_ground_truths, greedy_preds)
    print(f"\nRAW MODEL WER: {baseline_wer:.4f}")

    # 6. Grid Search for Alpha and Beta
    best_wer = 1.0
    best_params = (0, 0)
    
    # Expanded ranges to find the true minimum
    alphas = [0.4, 0.45, 0.5, 0.55, 0.6] 
    betas = [2.5, 2.75, 3.0, 3.25, 3.5]

    print(f"\n--- Step 2: Grid Search ({len(alphas) * len(betas)} combinations) ---")
    print(f"{'Alpha':<8} | {'Beta':<8} | {'WER':<10}")
    print("-" * 35)

    # Initialize decoder ONCE to save time and memory
    decoder = build_ctcdecoder(
        labels=sorted_vocab,
        kenlm_model_path=LM_PATH,
        unigrams=UNIGRAM_LIST,
    )

    for a in alphas:
        for b in betas:
            # Update the existing decoder's weights (fast)
            decoder.reset_params(alpha=a, beta=b)
            
            # Run the decoding
            kenlm_preds = [decoder.decode(l).lower() for l in all_logits_numpy]
            current_wer = wer(all_ground_truths, kenlm_preds)
            
            print(f"{a:<8} | {b:<8} | {current_wer:.4f}")
            
            if current_wer < best_wer:
                best_wer = current_wer
                best_params = (a, b)
    
    return best_params

Extracting unigrams...


#### Trigram Refinement

In [18]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [03:16<00:00,  9.57it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1237

--- Step 2: Grid Search (25 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.4      | 2.5      | 0.1074
0.4      | 2.75     | 0.1083
0.4      | 3.0      | 0.1095
0.4      | 3.25     | 0.1107
0.4      | 3.5      | 0.1120
0.45     | 2.5      | 0.1067
0.45     | 2.75     | 0.1074
0.45     | 3.0      | 0.1079
0.45     | 3.25     | 0.1093
0.45     | 3.5      | 0.1103
0.5      | 2.5      | 0.1064
0.5      | 2.75     | 0.1067
0.5      | 3.0      | 0.1064
0.5      | 3.25     | 0.1075
0.5      | 3.5      | 0.1088
0.55     | 2.5      | 0.1074
0.55     | 2.75     | 0.1072
0.55     | 3.0      | 0.1071
0.55     | 3.25     | 0.1074
0.55     | 3.5      | 0.1076
0.6      | 2.5      | 0.1086
0.6      | 2.75     | 0.1085
0.6      | 3.0      | 0.1078
0.6      | 3.25     | 0.1077
0.6      | 3.5      | 0.1079


#### Bi grams

In [20]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [03:25<00:00,  9.18it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1237

--- Step 2: Grid Search (30 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.3      | 1.0      | 0.1074
0.3      | 2.0      | 0.1083
0.3      | 3.0      | 0.1132
0.3      | 4.0      | 0.1243
0.3      | 5.0      | 0.1430
0.3      | 7.0      | 0.1960
0.5      | 1.0      | 0.1097
0.5      | 2.0      | 0.1071
0.5      | 3.0      | 0.1070
0.5      | 4.0      | 0.1123
0.5      | 5.0      | 0.1215
0.5      | 7.0      | 0.1584
0.7      | 1.0      | 0.1161
0.7      | 2.0      | 0.1116
0.7      | 3.0      | 0.1087
0.7      | 4.0      | 0.1089
0.7      | 5.0      | 0.1100
0.7      | 7.0      | 0.1333
0.9      | 1.0      | 0.1317
0.9      | 2.0      | 0.1241
0.9      | 3.0      | 0.1182
0.9      | 4.0      | 0.1150
0.9      | 5.0      | 0.1125
0.9      | 7.0      | 0.1169
1.1      | 1.0      | 0.1549
1.1      | 2.0      | 0.1436
1.1      | 3.0      | 0.1346
1.1      | 4.0      | 0.1269
1.1      | 5.0      | 0.1229
1.1      | 7.0      

#### Four Gram Big Model

In [19]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:53<00:00, 10.86it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1237

--- Step 2: Grid Search (30 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.3      | 1.0      | 0.1072
0.3      | 2.0      | 0.1086
0.3      | 3.0      | 0.1133
0.3      | 4.0      | 0.1277
0.3      | 5.0      | 0.1467
0.3      | 7.0      | 0.1974
0.5      | 1.0      | 0.1092
0.5      | 2.0      | 0.1071
0.5      | 3.0      | 0.1075
0.5      | 4.0      | 0.1132
0.5      | 5.0      | 0.1245
0.5      | 7.0      | 0.1658
0.7      | 1.0      | 0.1169
0.7      | 2.0      | 0.1123
0.7      | 3.0      | 0.1106
0.7      | 4.0      | 0.1098
0.7      | 5.0      | 0.1121
0.7      | 7.0      | 0.1408
0.9      | 1.0      | 0.1335
0.9      | 2.0      | 0.1243
0.9      | 3.0      | 0.1194
0.9      | 4.0      | 0.1145
0.9      | 5.0      | 0.1118
0.9      | 7.0      | 0.1206
1.1      | 1.0      | 0.1571
1.1      | 2.0      | 0.1438
1.1      | 3.0      | 0.1348
1.1      | 4.0      | 0.1280
1.1      | 5.0      | 0.1240
1.1      | 7.0      

### Trigram Big Model

In [21]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:54<00:00, 10.80it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1237

--- Step 2: Grid Search (30 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.3      | 1.0      | 0.1077
0.3      | 2.0      | 0.1087
0.3      | 3.0      | 0.1130
0.3      | 4.0      | 0.1280
0.3      | 5.0      | 0.1467
0.3      | 7.0      | 0.1956
0.5      | 1.0      | 0.1093
0.5      | 2.0      | 0.1072
0.5      | 3.0      | 0.1064
0.5      | 4.0      | 0.1116
0.5      | 5.0      | 0.1257
0.5      | 7.0      | 0.1660
0.7      | 1.0      | 0.1166
0.7      | 2.0      | 0.1114
0.7      | 3.0      | 0.1103
0.7      | 4.0      | 0.1089
0.7      | 5.0      | 0.1123
0.7      | 7.0      | 0.1407
0.9      | 1.0      | 0.1324
0.9      | 2.0      | 0.1248
0.9      | 3.0      | 0.1179
0.9      | 4.0      | 0.1141
0.9      | 5.0      | 0.1127
0.9      | 7.0      | 0.1207
1.1      | 1.0      | 0.1581
1.1      | 2.0      | 0.1451
1.1      | 3.0      | 0.1354
1.1      | 4.0      | 0.1283
1.1      | 5.0      | 0.1249
1.1      | 7.0      

### Fourgram Mid Reduction

In [20]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:57<00:00, 10.62it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1724

--- Step 2: Grid Search (30 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.3      | 1.0      | 0.1378
0.3      | 2.0      | 0.1402
0.3      | 3.0      | 0.1483
0.3      | 4.0      | 0.1612
0.3      | 5.0      | 0.1869
0.3      | 7.0      | 0.2542
0.5      | 1.0      | 0.1337
0.5      | 2.0      | 0.1329
0.5      | 3.0      | 0.1349
0.5      | 4.0      | 0.1422
0.5      | 5.0      | 0.1576
0.5      | 7.0      | 0.2099
0.7      | 1.0      | 0.1361
0.7      | 2.0      | 0.1347
0.7      | 3.0      | 0.1326
0.7      | 4.0      | 0.1334
0.7      | 5.0      | 0.1392
0.7      | 7.0      | 0.1759
0.9      | 1.0      | 0.1430
0.9      | 2.0      | 0.1394
0.9      | 3.0      | 0.1370
0.9      | 4.0      | 0.1341
0.9      | 5.0      | 0.1343
0.9      | 7.0      | 0.1526
1.1      | 1.0      | 0.1541
1.1      | 2.0      | 0.1485
1.1      | 3.0      | 0.1434
1.1      | 4.0      | 0.1396
1.1      | 5.0      | 0.1377
1.1      | 7.0      

### Bigrams Mid Reduction

In [19]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:51<00:00, 10.98it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1724

--- Step 2: Grid Search (30 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.3      | 1.0      | 0.1388
0.3      | 2.0      | 0.1418
0.3      | 3.0      | 0.1497
0.3      | 4.0      | 0.1634
0.3      | 5.0      | 0.1871
0.3      | 7.0      | 0.2546
0.5      | 1.0      | 0.1348
0.5      | 2.0      | 0.1335
0.5      | 3.0      | 0.1357
0.5      | 4.0      | 0.1437
0.5      | 5.0      | 0.1560
0.5      | 7.0      | 0.2085
0.7      | 1.0      | 0.1380
0.7      | 2.0      | 0.1351
0.7      | 3.0      | 0.1335
0.7      | 4.0      | 0.1353
0.7      | 5.0      | 0.1409
0.7      | 7.0      | 0.1773
0.9      | 1.0      | 0.1451
0.9      | 2.0      | 0.1406
0.9      | 3.0      | 0.1384
0.9      | 4.0      | 0.1348
0.9      | 5.0      | 0.1346
0.9      | 7.0      | 0.1508
1.1      | 1.0      | 0.1557
1.1      | 2.0      | 0.1489
1.1      | 3.0      | 0.1447
1.1      | 4.0      | 0.1409
1.1      | 5.0      | 0.1387
1.1      | 7.0      

#### Trigrams Mid Reduction

In [22]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:58<00:00, 10.54it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1724

--- Step 2: Grid Search (30 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.3      | 1.0      | 0.1384
0.3      | 2.0      | 0.1405
0.3      | 3.0      | 0.1485
0.3      | 4.0      | 0.1608
0.3      | 5.0      | 0.1849
0.3      | 7.0      | 0.2533
0.5      | 1.0      | 0.1334
0.5      | 2.0      | 0.1327
0.5      | 3.0      | 0.1348
0.5      | 4.0      | 0.1425
0.5      | 5.0      | 0.1568
0.5      | 7.0      | 0.2066
0.7      | 1.0      | 0.1354
0.7      | 2.0      | 0.1341
0.7      | 3.0      | 0.1326
0.7      | 4.0      | 0.1336
0.7      | 5.0      | 0.1402
0.7      | 7.0      | 0.1750
0.9      | 1.0      | 0.1430
0.9      | 2.0      | 0.1382
0.9      | 3.0      | 0.1362
0.9      | 4.0      | 0.1341
0.9      | 5.0      | 0.1348
0.9      | 7.0      | 0.1524
1.1      | 1.0      | 0.1546
1.1      | 2.0      | 0.1489
1.1      | 3.0      | 0.1440
1.1      | 4.0      | 0.1408
1.1      | 5.0      | 0.1382
1.1      | 7.0      

#### Redcued, fourgram

In [12]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [03:26<00:00,  9.12it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1722

--- Step 2: Grid Search (30 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.3      | 1.0      | 0.1375
0.3      | 2.0      | 0.1408
0.3      | 3.0      | 0.1492
0.3      | 4.0      | 0.1652
0.3      | 5.0      | 0.1913
0.3      | 7.0      | 0.2634
0.5      | 1.0      | 0.1335
0.5      | 2.0      | 0.1324
0.5      | 3.0      | 0.1351
0.5      | 4.0      | 0.1444
0.5      | 5.0      | 0.1625
0.5      | 7.0      | 0.2159
0.7      | 1.0      | 0.1359
0.7      | 2.0      | 0.1334
0.7      | 3.0      | 0.1316
0.7      | 4.0      | 0.1329
0.7      | 5.0      | 0.1401
0.7      | 7.0      | 0.1815
0.9      | 1.0      | 0.1418
0.9      | 2.0      | 0.1388
0.9      | 3.0      | 0.1352
0.9      | 4.0      | 0.1330
0.9      | 5.0      | 0.1337
0.9      | 7.0      | 0.1559
1.1      | 1.0      | 0.1542
1.1      | 2.0      | 0.1496
1.1      | 3.0      | 0.1442
1.1      | 4.0      | 0.1396
1.1      | 5.0      | 0.1372
1.1      | 7.0      

#### Reduced, trigram, actual split, tokenised + unigrams

In [14]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [03:18<00:00,  9.49it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



RAW MODEL WER: 0.1722

--- Step 2: Grid Search (30 combinations) ---
Alpha    | Beta     | WER       
-----------------------------------
0.3      | 1.0      | 0.1374
0.3      | 2.0      | 0.1406
0.3      | 3.0      | 0.1479
0.3      | 4.0      | 0.1649
0.3      | 5.0      | 0.1925
0.3      | 7.0      | 0.2605
0.5      | 1.0      | 0.1331
0.5      | 2.0      | 0.1328
0.5      | 3.0      | 0.1358
0.5      | 4.0      | 0.1433
0.5      | 5.0      | 0.1607
0.5      | 7.0      | 0.2135
0.7      | 1.0      | 0.1361
0.7      | 2.0      | 0.1337
0.7      | 3.0      | 0.1314
0.7      | 4.0      | 0.1333
0.7      | 5.0      | 0.1399
0.7      | 7.0      | 0.1814
0.9      | 1.0      | 0.1425
0.9      | 2.0      | 0.1390
0.9      | 3.0      | 0.1357
0.9      | 4.0      | 0.1333
0.9      | 5.0      | 0.1339
0.9      | 7.0      | 0.1563
1.1      | 1.0      | 0.1532
1.1      | 2.0      | 0.1488
1.1      | 3.0      | 0.1440
1.1      | 4.0      | 0.1401
1.1      | 5.0      | 0.1369
1.1      | 7.0      

##### Reduced Vocabulary with actual spliuts, tokenised and 4 grams.

In [17]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:57<00:00, 10.61it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



RAW MODEL WER: 0.1722

Alpha    | Beta     | WER       
-----------------------------------


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.85     | 2.0      | 0.1429
0.85     | 3.0      | 0.1398


...

0.9      | 3.0      | 0.1413
0.9      | 4.0      | 0.1402


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 2.0      | 0.1465
0.95     | 3.0      | 0.1433


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 4.0      | 0.1417

--- Optimization Result ---
Best Alpha: 0.85
Best Beta:  4.0
Best WER:   0.1394 (Reduction of 0.0328)


#### W2v tokenised but with actual data splits

In [24]:
best_a, best_b = optimize_with_dataset(common_voice_dev)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:50<00:00, 11.04it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



RAW MODEL WER: 0.1709

Alpha    | Beta     | WER       
-----------------------------------


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.85     | 2.0      | 0.1435
0.85     | 3.0      | 0.1401


...

0.95     | 2.0      | 0.1468


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 3.0      | 0.1442


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 4.0      | 0.1409

--- Optimization Result ---
Best Alpha: 0.85
Best Beta:  4.0
Best WER:   0.1384 (Reduction of 0.0325)


#### Standard w2v2-bert + LM

In [ ]:
best_a, best_b = optimize_with_dataset(common_voice_test)

--- Step 1: Generating Logits for 1660 samples ---


100%|██████████| 1660/1660 [02:36<00:00, 10.60it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



RAW MODEL WER: 0.1390

Alpha    | Beta     | WER       
-----------------------------------


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.85     | 2.0      | 0.0979
0.85     | 3.0      | 0.0974


...

0.9      | 4.0      | 0.0980


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 2.0      | 0.0996


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 3.0      | 0.0992
0.95     | 4.0      | 0.0979

--- Optimization Result ---
Best Alpha: 0.9
Best Beta:  3.0
Best WER:   0.0973 (Reduction of 0.0417)


#### Tokenised w2v2-bert + lm

In [17]:
best_a, best_b = optimize_with_dataset(common_voice_test)

--- Step 1: Generating Logits for 1660 samples ---


100%|██████████| 1660/1660 [02:35<00:00, 10.67it/s]



RAW MODEL WER: 0.1256

Alpha    | Beta     | WER       
-----------------------------------


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.85     | 2.0      | 0.0886
0.85     | 3.0      | 0.0880


...

0.9      | 4.0      | 0.0881
0.95     | 2.0      | 0.0898


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 3.0      | 0.0886


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 4.0      | 0.0878

--- Optimization Result ---
Best Alpha: 0.95
Best Beta:  4.0
Best WER:   0.0878 (Reduction of 0.0378)


### Tokenised + Processed

In [15]:
best_a, best_b = optimize_with_dataset(common_voice_test)

--- Step 1: Generating Logits for 1660 samples ---


100%|██████████| 1660/1660 [02:58<00:00,  9.32it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



RAW MODEL WER: 0.1256

Alpha    | Beta     | WER       
-----------------------------------
0.85     | 2.0      | 0.0889


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.85     | 3.0      | 0.0880


...

0.9      | 4.0      | 0.0882


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 2.0      | 0.0896


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 3.0      | 0.0885
0.95     | 4.0      | 0.0878

--- Optimization Result ---
Best Alpha: 0.95
Best Beta:  4.0
Best WER:   0.0878 (Reduction of 0.0378)


### Vanilla CV+MASRI W2v2 + Highly Processed KenLM

In [27]:
# Run with the new kenlm. The rest used the older one
best_a, best_b = optimize_with_dataset(common_voice_test)

--- Step 1: Generating Logits for 1660 samples ---


100%|██████████| 1660/1660 [03:06<00:00,  8.90it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



RAW MODEL WER: 0.1141

Alpha    | Beta     | WER       
-----------------------------------


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.85     | 2.0      | 0.0808


...

0.9      | 3.0      | 0.0808


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.9      | 4.0      | 0.0812
0.95     | 2.0      | 0.0814


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 3.0      | 0.0809
0.95     | 4.0      | 0.0812

--- Optimization Result ---
Best Alpha: 0.85
Best Beta:  3.0
Best WER:   0.0807 (Reduction of 0.0334)


In [38]:
# Run the process
best_a, best_b = optimize_with_dataset(common_voice_test)

--- Step 1: Generating Logits for 1660 samples ---


100%|██████████| 1660/1660 [02:39<00:00, 10.41it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



RAW MODEL WER: 0.1141

Alpha    | Beta     | WER       
-----------------------------------


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.85     | 2.0      | 0.0807
0.85     | 3.0      | 0.0805


...

0.95     | 2.0      | 0.0812


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 3.0      | 0.0806


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 4.0      | 0.0809

--- Optimization Result ---
Best Alpha: 0.85
Best Beta:  3.0
Best WER:   0.0805 (Reduction of 0.0336)


In [36]:
# Run the optimization
train_subset = common_voice_train.shuffle(seed=42).select(range(2000))

# Run the same optimization function on this subset
best_alpha_train, best_beta_train = optimize_with_dataset(train_subset)

print(f"Parameters tuned on Train: Alpha {best_alpha_train}, Beta {best_beta_train}")

--- Step 1: Generating Logits for 2000 samples ---


100%|██████████| 2000/2000 [02:52<00:00, 11.62it/s]



RAW MODEL WER: 0.0370

Alpha    | Beta     | WER       
-----------------------------------


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.85     | 2.0      | 0.0396


...

0.95     | 2.0      | 0.0411


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 3.0      | 0.0407


Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.


0.95     | 4.0      | 0.0404

--- Optimization Result ---
Best Alpha: 0.85
Best Beta:  4.0
Best WER:   0.0393 (Reduction of -0.0023)
Parameters tuned on Train: Alpha 0.85, Beta 4.0


## Other Experiments

#### Raw Model

In [40]:
import torch
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from jiwer import wer, cer  # <--- Added cer import
from tqdm import tqdm

# 1. Setup
base_path = Path(".") 
my_model_path = base_path / "Runs" / "checkpoint-5690"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load Model and Processor
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

def evaluate_raw_model(dataset):
    all_ground_truths = []
    greedy_preds = []

    print(f"--- Evaluating {len(dataset)} samples (Greedy Decoding) ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        # Preprocess
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        # Inference
        with torch.no_grad():
            logits = model(input_features).logits
        
        # Greedy Decode (Argmax)
        pred_ids = torch.argmax(logits, dim=-1)
        transcription = processor.batch_decode(pred_ids)[0]
        
        # Store results
        greedy_preds.append(transcription.lower())
        all_ground_truths.append(batch["sentence"].lower())

    # --- Metrics Calculation ---
    final_wer = wer(all_ground_truths, greedy_preds)
    final_cer = cer(all_ground_truths, greedy_preds)
    
    print("\n" + "="*35)
    print(f"RAW MODEL RESULTS:")
    print(f"WER: {final_wer:.4f} ({final_wer*100:.2f}%)")
    print(f"CER: {final_cer:.4f} ({final_cer*100:.2f}%)")
    print("="*35)
    
    # Sanity Check
    print("\nSample Transcriptions:")
    for j in range(min(2, len(all_ground_truths))):
        print(f"REF: {all_ground_truths[j]}")
        print(f"HYP: {greedy_preds[j]}")
        print("-" * 10)

    return {"wer": final_wer, "cer": final_cer}

In [ ]:
import torch
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from jiwer import wer, cer
from tqdm import tqdm
import malti.tokeniser  # Import the malti library

# 1. Setup
base_path = Path(".") 
my_model_path = base_path / "Runs" / "SplitRunReduced" / "checkpoint-2780"
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize the Maltese Detokenizer
malti_tokenizer = malti.tokeniser.KMTokeniser()

# 2. Load Model and Processor
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

def evaluate_raw_model(dataset, print_all=False):
    all_ground_truths = []
    greedy_preds = []
    mismatches = []  # List to store errors for review

    print(f"--- Evaluating {len(dataset)} samples ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            logits = model(input_features).logits
        
        pred_ids = torch.argmax(logits, dim=-1)
        raw_transcription = processor.batch_decode(pred_ids)[0].lower()
        raw_reference = batch["sentence"].lower()

        # Detokenization
        detok_transcription = malti_tokenizer.detokenise(raw_transcription.split())
        detok_reference = malti_tokenizer.detokenise(raw_reference.split())
        
        greedy_preds.append(detok_transcription)
        all_ground_truths.append(detok_reference)

        # Track Mismatches
        if detok_transcription != detok_reference:
            mismatches.append({
                "index": i,
                "ref": detok_reference,
                "hyp": detok_transcription
            })
        
        # Optional: Print every single result as it happens
        if print_all:
            print(f"[{i}] REF: {detok_reference}")
            print(f"[{i}] HYP: {detok_transcription}")
            print("-" * 20)

    # --- Metrics ---
    final_wer = wer(all_ground_truths, greedy_preds)
    final_cer = cer(all_ground_truths, greedy_preds)
    
    # --- Final Report ---
    print("\n" + "="*40)
    print(f"RESULTS Summary")
    print(f"Total Samples: {len(dataset)}")
    print(f"Perfect Matches: {len(dataset) - len(mismatches)}")
    print(f"Mismatches: {len(mismatches)}")
    print(f"WER: {final_wer:.4f} | CER: {final_cer:.4f}")
    print("="*40)
    
    if len(mismatches) > 0 and not print_all:
        print("\n--- DETAILED MISMATCHES ---")
        for m in mismatches:
            print(f"Sample #{m['index']}")
            print(f"  REF: {m['ref']}")
            print(f"  HYP: {m['hyp']}")
            print(f"  diff: {'[Length Mismatch]' if len(m['ref']) != len(m['hyp']) else ''}")
            print("-" * 15)

    return {"wer": final_wer, "cer": final_cer, "mismatches": mismatches}

Extracting unigrams and building decoder...


Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x77a51045b610>>
Traceback (most recent call last):
  File "/home/ian/miniconda3/envs/w2v2_KenLM/lib/python3.9/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


In [21]:
import torch
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer
from tqdm import tqdm
import malti.tokeniser

# 1. Setup
base_path = Path(".") 
my_model_path = base_path / "Runs" / "SplitRunReducedButMid" / "checkpoint-2780"
LM_PATH = str(base_path / "4_tok_red.bin")
ARPA_PATH = str(base_path / "3_tok_red.arpa")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize Maltese Detokenizer
malti_tokenizer = malti.tokeniser.KMTokeniser()

# 2. Load Model and Processor
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

# 3. Prepare Decoder Vocab and Unigrams
vocab = processor.tokenizer.get_vocab()
sorted_vocab = [k for k, v in sorted(vocab.items(), key=lambda item: item[1])]

def get_unigram_list(arpa_path):
    unigrams = []
    with open(arpa_path, "r") as f:
        is_unigram_section = False
        for line in f:
            if "\\1-grams:" in line:
                is_unigram_section = True
                continue
            if "\\2-grams:" in line or line.startswith("\\end\\"):
                break
            if is_unigram_section:
                parts = line.strip().split()
                if len(parts) >= 2:
                    word = parts[1]
                    if word not in ["<s>", "</s>", "<unk>"]:
                        unigrams.append(word)
    return unigrams

print("Extracting unigrams...")
unigrams = get_unigram_list(ARPA_PATH)

def evaluate_with_kenlm(dataset, alpha=0.85, beta=4.0):
    all_ground_truths = []
    greedy_preds = []
    kenlm_preds = []
    
    # Initialize Decoder
    decoder = build_ctcdecoder(
        labels=sorted_vocab,
        kenlm_model_path=LM_PATH,
        unigrams=unigrams,
        alpha=alpha,
        beta=beta
    )

    print(f"--- Evaluating {len(dataset)} samples with KenLM ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            logits = model(input_features).logits
        
        # 1. Greedy Decoding
        pred_ids = torch.argmax(logits, dim=-1)
        raw_greedy = processor.batch_decode(pred_ids)[0].lower()
        
        # 2. KenLM Decoding (Logits need to be CPU numpy for pyctcdecode)
        logits_numpy = logits.cpu().numpy()[0]
        raw_kenlm = decoder.decode(logits_numpy).lower()

        # 3. Reference
        raw_reference = batch["sentence"].lower()

        # --- Detokenization (The "Fair" Step) ---
        detok_ref = malti_tokenizer.detokenise(raw_reference.split())
        detok_greedy = malti_tokenizer.detokenise(raw_greedy.split())
        detok_kenlm = malti_tokenizer.detokenise(raw_kenlm.split())
        
        all_ground_truths.append(detok_ref)
        greedy_preds.append(detok_greedy)
        kenlm_preds.append(detok_kenlm)

    # --- Metrics Calculation on Detokenized Strings ---
    base_wer = wer(all_ground_truths, greedy_preds)
    base_cer = cer(all_ground_truths, greedy_preds)
    
    k_wer = wer(all_ground_truths, kenlm_preds)
    k_cer = cer(all_ground_truths, kenlm_preds)

    # --- Final Report ---
    print("\n" + "="*50)
    print(f"{'Metric (Detokenized)':<20} | {'Raw Model':<12} | {'With KenLM':<12}")
    print("-" * 50)
    print(f"{'WER':<20} | {base_wer:<12.4f} | {k_wer:<12.4f}")
    print(f"{'CER':<20} | {base_cer:<12.4f} | {k_cer:<12.4f}")
    print("="*50)
    
    return {"greedy_wer": base_wer, "kenlm_wer": k_wer}


Extracting unigrams...


#### Evaluating impact of n-grams

In [22]:
#Fourgram
evaluate_with_kenlm(common_voice_dev,alpha=0.7, beta=3.0) 

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


--- Evaluating 1883 samples with KenLM ---


100%|██████████| 1883/1883 [05:56<00:00,  5.28it/s]



Metric (Detokenized) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.1989       | 0.1530      
CER                  | 0.0516       | 0.0477      


{'greedy_wer': 0.19887904538058218, 'kenlm_wer': 0.1529560658108841}

In [20]:
#Bigrams
evaluate_with_kenlm(common_voice_dev,alpha=0.5, beta=2.0) 

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


--- Evaluating 1883 samples with KenLM ---


100%|██████████| 1883/1883 [03:09<00:00,  9.95it/s]


Metric (Detokenized) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.1989       | 0.1549      
CER                  | 0.0516       | 0.0469      


{'greedy_wer': 0.19887904538058218, 'kenlm_wer': 0.1548845898873019}

#### Detokenised on Test sets

In [20]:
# Common Voice
evaluate_with_kenlm(common_voice_test,alpha=0.7, beta=3.0) 

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


--- Evaluating 1224 samples with KenLM ---


100%|██████████| 1224/1224 [01:55<00:00, 10.62it/s]



Metric (Detokenized) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.1001       | 0.0699      
CER                  | 0.0222       | 0.0173      


{'greedy_wer': 0.10009225092250923, 'kenlm_wer': 0.069880073800738}

In [19]:
# MASRI Only
evaluate_with_kenlm(common_voice_test,alpha=0.7, beta=3.0) 

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


--- Evaluating 668 samples with KenLM ---


100%|██████████| 668/668 [01:44<00:00,  6.40it/s]



Metric (Detokenized) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.3341       | 0.2669      
CER                  | 0.0979       | 0.0948      


{'greedy_wer': 0.33408352088022003, 'kenlm_wer': 0.26694173543385846}

In [20]:
# Combined
evaluate_with_kenlm(common_voice_test,alpha=0.7, beta=3.0) 

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


--- Evaluating 1892 samples with KenLM ---


100%|██████████| 1892/1892 [04:10<00:00,  7.54it/s]



Metric (Detokenized) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.2124       | 0.1644      
CER                  | 0.0581       | 0.0541      


{'greedy_wer': 0.21235752849430115, 'kenlm_wer': 0.1644271145770846}

#### Vocab tests

In [26]:
# Split run (uncleaned vocab or barely)
evaluate_with_kenlm(common_voice_dev,alpha=0.7, beta=3.0) 

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


--- Evaluating 1883 samples with KenLM ---


100%|██████████| 1883/1883 [06:35<00:00,  4.76it/s]



Metric (Detokenized) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.1979       | 0.1536      
CER                  | 0.0518       | 0.0483      


{'greedy_wer': 0.19791478334237328, 'kenlm_wer': 0.15361899596215273}

In [24]:
# Mid Vocabulary (final vocab used)
evaluate_with_kenlm(common_voice_dev,alpha=0.7, beta=3.0) 

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


--- Evaluating 1883 samples with KenLM ---


100%|██████████| 1883/1883 [04:31<00:00,  6.94it/s]



Metric (Detokenized) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.1989       | 0.1526      
CER                  | 0.0516       | 0.0478      


{'greedy_wer': 0.19887904538058218, 'kenlm_wer': 0.15259446754655578}

In [21]:
# Split Run reduced
evaluate_with_kenlm(common_voice_dev,alpha=0.7, beta=3.0) 

Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?


--- Evaluating 1883 samples with KenLM ---


100%|██████████| 1883/1883 [04:12<00:00,  7.45it/s]



Metric (Detokenized) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.2012       | 0.1546      
CER                  | 0.0524       | 0.0481      


{'greedy_wer': 0.20122943409871633, 'kenlm_wer': 0.1545832580003616}

#### Big Model - Common Voice Only

In [41]:
evaluate_raw_model(common_voice_test) 

--- Evaluating 1224 samples (Greedy Decoding) ---


100%|██████████| 1224/1224 [02:05<00:00,  9.74it/s]


RAW MODEL RESULTS:
WER: 0.0546 (5.46%)
CER: 0.0117 (1.17%)

Sample Transcriptions:
REF: dan ma sar qatt
HYP: dan ma sar qatt
----------
REF: min jaf kif u biex se jimlewha
HYP: min jaf kif u biex se jimlewha
----------


{'wer': 0.054610255231087604, 'cer': 0.011733092996862274}

#### Big Model - MASRI Only

In [23]:
evaluate_raw_model(common_voice_test) 

--- Evaluating 668 samples (Greedy Decoding) ---


100%|██████████| 668/668 [01:11<00:00,  9.29it/s]


RAW MODEL RESULTS:
WER: 0.2262 (22.62%)
CER: 0.0729 (7.29%)

Sample Transcriptions:
REF: maskarat tini perlina għax warajk għandek xadina minflok waħda tini tnejn biex warajk ma jkollok xejn
HYP: maskerat tini perlina għax warajt għandek xadina minflok waħda tini tnejn biex warajt ma jkollok xejn
----------
REF: eħe f' dan is-suġġett se nagħtu ħarsa lejn in-novelli u r-rumanzi tal-awturi maltin kosmopolitani jiġifieri li bdew jiktbu
HYP: eħe f' dan is-suġġett se nagħtu ħarsa lejn in-novelli u r-rumanzi tal-awturi maltin kosmopolitan jiġifieri li bdew jiktbu
----------


{'wer': 0.22616845480330497, 'cer': 0.07290320197953983}

##### Bid model

In [23]:
evaluate_raw_model(common_voice_test) 

--- Evaluating 1892 samples (Greedy Decoding) ---


100%|██████████| 1892/1892 [03:11<00:00,  9.88it/s]



RAW MODEL RESULTS:
WER: 0.1374 (13.74%)
CER: 0.0408 (4.08%)

Sample Transcriptions:
REF: maskarat tini perlina għax warajk għandek xadina minflok waħda tini tnejn biex warajk ma jkollok xejn
HYP: maskerat tini perlina għax warajt għandek xadina minflok waħda tini tnejn biex warajt ma jkollok xejn
----------
REF: eħe f' dan is-suġġett se nagħtu ħarsa lejn in-novelli u r-rumanzi tal-awturi maltin kosmopolitani jiġifieri li bdew jiktbu
HYP: eħe f' dan is-suġġett se nagħtu ħarsa lejn in-novelli u r-rumanzi tal-awturi maltin kosmopolitan jiġifieri li bdew jiktbu
----------


{'wer': 0.13738323317665257, 'cer': 0.040773209432804415}

##### Mid test for sanity.

In [33]:
evaluate_raw_model(common_voice_test) 

--- Evaluating 1892 samples (Greedy Decoding) ---


100%|██████████| 1892/1892 [02:58<00:00, 10.60it/s]



RAW MODEL RESULTS:
WER: 0.1816 (18.16%)
CER: 0.0575 (5.75%)

Sample Transcriptions:
REF: maskarat tini perlina għax warajk għandek xadina minflok waħda tini tnejn biex warajk ma jkollok xejn
HYP: maskarat ini perlina għax warajk għandek x' għdina minflok waħda tini tnejn ix warajk ma jkollok xejn
----------
REF: eħe f' dan is- suġġett se nagħtu ħarsa lejn in- novelli u r- rumanzi tal- awturi maltin kosmopolitani jiġifieri li bdew jiktbu
HYP: eħe f' dan is- suġġett se nagħtu ħarsa lejn n- novelli u r- rumanzi ta' l- awturi maltin kosmopolitan jiġifieri bdew jiktbu
----------


{'wer': 0.18158334599604922, 'cer': 0.05749326951839665}

##### Split Run Reduced Tokenised

In [24]:
evaluate_raw_model(common_voice_test) 

--- Evaluating 1892 samples ---


100%|██████████| 1892/1892 [02:51<00:00, 11.02it/s]


RESULTS Summary
Total Samples: 1892
Perfect Matches: 776
Mismatches: 1116
WER: 0.2099 | CER: 0.0580

--- DETAILED MISMATCHES ---
Sample #0
  REF: maskarat tini perlina għax warajk għandek xadina minflok waħda tini tnejn biex warajk ma jkollok xejn
  HYP: maskarat ini perlina għax warajk għandek x'għdina minflok waħda tini tnejn ix warajk ma jkollok xejn
  diff: [Length Mismatch]
---------------
Sample #1
  REF: eħe f'dan is-suġġett se nagħtu ħarsa lejn in-novelli u r-rumanzi tal-awturi maltin kosmopolitani jiġifieri li bdew jiktbu
  HYP: eħe f'dan is-suġġett se nagħtu ħarsa lejn n-novelli u r-rumanzi ta' l-awturi maltin kosmopolitan jiġifieri bdew jiktbu
  diff: [Length Mismatch]
---------------
Sample #2
  REF: mis-snin disgħin tas-seklu għoxrin issa meta ngħid kosmopolitani xi rrid infisser
  HYP: mis-snin disgħin tas-seklu għoxrin issa meta ngħid kosmopolitani x'irrid infisser
  diff: 
---------------
Sample #3
  REF: mela e kosmopolitani dawn l-awturi warrbu l-fruntieri
  HYP: mel

{'wer': 0.20989802039592081,
 'cer': 0.05800828564406839,
 'mismatches': [{'index': 0,
   'ref': 'maskarat tini perlina għax warajk għandek xadina minflok waħda tini tnejn biex warajk ma jkollok xejn',
   'hyp': "maskarat ini perlina għax warajk għandek x'għdina minflok waħda tini tnejn ix warajk ma jkollok xejn"},
  {'index': 1,
   'ref': "eħe f'dan is-suġġett se nagħtu ħarsa lejn in-novelli u r-rumanzi tal-awturi maltin kosmopolitani jiġifieri li bdew jiktbu",
   'hyp': "eħe f'dan is-suġġett se nagħtu ħarsa lejn n-novelli u r-rumanzi ta' l-awturi maltin kosmopolitan jiġifieri bdew jiktbu"},
  {'index': 2,
   'ref': 'mis-snin disgħin tas-seklu għoxrin issa meta ngħid kosmopolitani xi rrid infisser',
   'hyp': "mis-snin disgħin tas-seklu għoxrin issa meta ngħid kosmopolitani x'irrid infisser"},
  {'index': 3,
   'ref': 'mela e kosmopolitani dawn l-awturi warrbu l-fruntieri',
   'hyp': "mela e b'kosmopolitan dawn l-awturi gwarrbu l-frontieri"},
  {'index': 4,
   'ref': 'u bdejna ma bqaj

##### Split Run Reduced UnTokenised

In [26]:
evaluate_raw_model(common_voice_test) 

--- Evaluating 1892 samples (Greedy Decoding) ---


100%|██████████| 1892/1892 [03:32<00:00,  8.91it/s]



RAW MODEL RESULTS:
WER: 0.2125 (21.25%)
CER: 0.0577 (5.77%)

Sample Transcriptions:
REF: maskarat tini perlina għax warajk għandek xadina minflok waħda tini tnejn biex warajk ma jkollok xejn
HYP: maskerat tini perlina għax warajk għandek xadina minflok waħda tini t-tnejn biex warajk ma jkollok xejn
----------
REF: eħe f' dan is-suġġett se nagħtu ħarsa lejn in-novelli u r-rumanzi tal-awturi maltin kosmopolitani jiġifieri li bdew jiktbu
HYP: eef dan is-suġġett se nagħtu ħarsa lejn in-novelli u r-rumanzi ta' l-awturi maltin kosmopolitan jiġifieri bdew jiktbu
----------


{'wer': 0.21247099422859522, 'cer': 0.05765757867975804}

##### Common Voice Tokenised but normalised

In [64]:
evaluate_raw_model(common_voice_test) 

--- Evaluating 1224 samples ---


100%|██████████| 1224/1224 [01:49<00:00, 11.16it/s]


RESULTS Summary
Total Samples: 1224
Perfect Matches: 866
Mismatches: 358
WER: 0.0710 | CER: 0.0151

--- DETAILED MISMATCHES ---
Sample #2
  REF: xi tħobb f'pajjiżek
  HYP: xi thobb f'pajjiżek
  diff: 
---------------
Sample #4
  REF: huwa suġġett għall-konfiska taħt dik il-proklama
  HYP: huwa suġġett tal-konfiska taħt dik il-proklama
  diff: [Length Mismatch]
---------------
Sample #8
  REF: isaac kien akkumpanjat mill-kowċ tiegħu u missieru alex bezzina
  HYP: aiżak kien ikkumpanjat mill-kowċ tiegħu u missieru alexs bezzina
  diff: [Length Mismatch]
---------------
Sample #14
  REF: din hija kompetizzjoni li tintlagħab fuq żewġ logħbiet kontra l-istess tim
  HYP: din hija kompetizzjoni li tintlagħab fuq żewġ lgħbiet kontra l-istess tim
  diff: [Length Mismatch]
---------------
Sample #15
  REF: ħalli ngħaddi issa għal dak li qed jingħad fir-rapport annwali
  HYP: ħalli ngħaddi iss għa dak li qed jingħad fir-rapport annwali
  diff: [Length Mismatch]
---------------
Sample #18
  REF: 

{'wer': 0.07103321033210332,
 'cer': 0.015077894104092571,
 'mismatches': [{'index': 2,
   'ref': "xi tħobb f'pajjiżek",
   'hyp': "xi thobb f'pajjiżek"},
  {'index': 4,
   'ref': 'huwa suġġett għall-konfiska taħt dik il-proklama',
   'hyp': 'huwa suġġett tal-konfiska taħt dik il-proklama'},
  {'index': 8,
   'ref': 'isaac kien akkumpanjat mill-kowċ tiegħu u missieru alex bezzina',
   'hyp': 'aiżak kien ikkumpanjat mill-kowċ tiegħu u missieru alexs bezzina'},
  {'index': 14,
   'ref': 'din hija kompetizzjoni li tintlagħab fuq żewġ logħbiet kontra l-istess tim',
   'hyp': 'din hija kompetizzjoni li tintlagħab fuq żewġ lgħbiet kontra l-istess tim'},
  {'index': 15,
   'ref': 'ħalli ngħaddi issa għal dak li qed jingħad fir-rapport annwali',
   'hyp': 'ħalli ngħaddi iss għa dak li qed jingħad fir-rapport annwali'},
  {'index': 18,
   'ref': "x'investors programme hu dan",
   'hyp': "x'investor s programm hu dan"},
  {'index': 20,
   'ref': 'clint camilleri jew therese comodini cachia',
   

##### Common Voise Untokenised

In [ ]:
evaluate_raw_model(common_voice_test) 

--- Evaluating 1224 samples (Greedy Decoding) ---


100%|██████████| 1224/1224 [01:52<00:00, 10.89it/s]


RAW MODEL RESULTS:
WER: 0.0667 (6.67%)
CER: 0.0141 (1.41%)

Sample Transcriptions:
REF: dan ma sar qatt
HYP: dan ma sar qatt
----------
REF: min jaf kif u biex se jimlewha
HYP: min jaf kif u biex se jimlewha
----------


{'wer': 0.0666819958611175, 'cer': 0.014053007543894786}

In [19]:
evaluate_raw_model(masri_test)

--- Evaluating 668 samples (Greedy Decoding) ---


100%|██████████| 668/668 [01:11<00:00,  9.28it/s]


RAW MODEL RESULTS:
WER: 0.2894 (28.94%)
CER: 0.0989 (9.89%)

Sample Transcriptions:
REF: maskarat tini perlina għax warajk għandek xadina minflok waħda tini tnejn biex warajk ma jkollok xejn
HYP: maskerattini perlina għax warajk għandek xadina minflok waħda tini - tnejn għiex warajk ma jkollok xejn
----------
REF: eħe f' dan is- suġġett se nagħtu ħarsa lejn in- novelli u r- rumanzi tal- awturi maltin kosmopolitani jiġifieri li bdew jiktbu
HYP: eħe f' dan is- suġġett se nagħtu ħarsa lejn in-novelli u r- rumanzi ta' l- awturi maltin kosmopolitan jiġifierili bdew jiktbu
----------


{'wer': 0.28939313984168863, 'cer': 0.0989193083573487}

#### With LM

In [19]:
import torch
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer
from tqdm import tqdm

# 1. Setup paths
base_path = Path(".") 
my_model_path = base_path / "Runs" / "SplitRunReducedNoToken" / "checkpoint-2780"
LM_PATH = str(base_path / "3_tok_red.bin")

device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load Model and Processor
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

# 3. Prepare Decoder Vocab
vocab = processor.tokenizer.get_vocab()
sorted_vocab = [k for k, v in sorted(vocab.items(), key=lambda item: item[1])]

def evaluate_kenlm_manual(dataset, alpha=0.85, beta=4.0):
    all_logits_numpy = []
    all_ground_truths = []
    greedy_preds = []

    print(f"--- Step 1: Generating Logits for {len(dataset)} samples ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            logits = model(input_features).logits
        
        all_logits_numpy.append(logits.cpu().numpy()[0])
        all_ground_truths.append(batch["sentence"].lower())
        
        pred_ids = torch.argmax(logits, dim=-1)
        greedy_preds.append(processor.batch_decode(pred_ids)[0].lower())

    # Calculate Raw Metrics
    baseline_wer = wer(all_ground_truths, greedy_preds)
    baseline_cer = cer(all_ground_truths, greedy_preds)

    # --- Step 2: Decode with fixed KenLM parameters ---
    print(f"\n--- Step 2: Decoding with KenLM (Alpha: {alpha}, Beta: {beta}) ---")
    
    decoder = build_ctcdecoder(
        labels=sorted_vocab,
        kenlm_model_path=LM_PATH,
        alpha=alpha,
        beta=beta
    )
    
    kenlm_preds = [decoder.decode(l).lower() for l in all_logits_numpy]
    kenlm_wer = wer(all_ground_truths, kenlm_preds)
    kenlm_cer = cer(all_ground_truths, kenlm_preds)

    # --- Final Results ---
    print("\n" + "="*45)
    print(f"{'Metric':<10} | {'Raw Model':<12} | {'With KenLM':<12}")
    print("-" * 45)
    print(f"{'WER':<10} | {baseline_wer:<12.4f} | {kenlm_wer:<12.4f}")
    print(f"{'CER':<10} | {baseline_cer:<12.4f} | {kenlm_cer:<12.4f}")
    print("="*45)
    
    improvement = baseline_wer - kenlm_wer
    print(f"\nWER Improvement: {improvement:.4f}")
    
    return {"raw_wer": baseline_wer, "kenlm_wer": kenlm_wer}

#### Big Run - COmbined but my test set not aidens.

In [20]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.5, beta=3.0)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [02:59<00:00, 10.51it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.2144       | 0.2138      
CER        | 0.0580       | 0.0610      

WER Improvement: 0.0006


#### Big Run - Common Voice

In [ ]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.5, beta=3.0)

##### Split Run Reduced Non-Tokenised

In [ ]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.7, beta=3.0)

##### SplitRun

In [19]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.7, beta=3.0)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [03:00<00:00, 10.46it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



--- Step 2: Decoding with KenLM (Alpha: 0.7, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1850       | 0.1546      
CER        | 0.0592       | 0.0590      

WER Improvement: 0.0304


##### Mid Reduction

In [21]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.7, beta=3.0)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [03:01<00:00, 10.42it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



--- Step 2: Decoding with KenLM (Alpha: 0.7, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1842       | 0.1503      
CER        | 0.0573       | 0.0563      

WER Improvement: 0.0339


##### High Reduction

In [23]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.7, beta=3.0)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [03:00<00:00, 10.50it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



--- Step 2: Decoding with KenLM (Alpha: 0.7, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1833       | 0.1525      
CER        | 0.0578       | 0.0577      

WER Improvement: 0.0308


#### With Unigrams

In [20]:
import torch
import time
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer
from tqdm import tqdm

# 1. Setup paths
base_path = Path(".") 
my_model_path = base_path / "Runs" / "checkpoint-5500"
LM_PATH = str(base_path / "3_untok_red.bin")
ARPA_PATH = str(base_path / "3_untok_red.arpa") # We need the text version for unigrams

device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Load Model and Processor
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

# 3. Prepare Decoder Vocab
vocab = processor.tokenizer.get_vocab()
sorted_vocab = [k for k, v in sorted(vocab.items(), key=lambda item: item[1])]

# --- NEW: Extract unigrams from ARPA to pass into the binary decoder ---
def get_unigram_list(arpa_path):
    unigrams = []
    with open(arpa_path, "r") as f:
        is_unigram_section = False
        for line in f:
            if "\\1-grams:" in line:
                is_unigram_section = True
                continue
            if "\\2-grams:" in line or line.startswith("\\end\\"):
                break
            if is_unigram_section:
                parts = line.strip().split()
                if len(parts) >= 2:
                    word = parts[1]
                    if word not in ["<s>", "</s>", "<unk>"]:
                        unigrams.append(word)
    return unigrams

print("Extracting unigrams...")
unigrams = get_unigram_list(ARPA_PATH)
# -----------------------------------------------------------------------

Extracting unigrams...


In [21]:
def evaluate_kenlm_manual(dataset, alpha=0.85, beta=4.0):
    all_logits_numpy = []
    all_ground_truths = []
    greedy_preds = []

    print(f"--- Step 1: Generating Logits for {len(dataset)} samples ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            logits = model(input_features).logits
        
        all_logits_numpy.append(logits.cpu().numpy()[0])
        all_ground_truths.append(batch["sentence"].lower())
        
        pred_ids = torch.argmax(logits, dim=-1)
        greedy_preds.append(processor.batch_decode(pred_ids)[0].lower())

    # Calculate Raw Metrics
    baseline_wer = wer(all_ground_truths, greedy_preds)
    baseline_cer = cer(all_ground_truths, greedy_preds)

    # --- Step 2: Decode with fixed KenLM parameters ---
    print(f"\n--- Step 2: Decoding with KenLM (Alpha: {alpha}, Beta: {beta}) ---")
    
    # UPDATED: Added 'unigrams' argument
    decoder = build_ctcdecoder(
        labels=sorted_vocab,
        kenlm_model_path=LM_PATH,
        unigrams=unigrams, # <--- Crucial fix
        alpha=alpha,
        beta=beta
    )
    
    kenlm_preds = [decoder.decode(l).lower() for l in all_logits_numpy]
    kenlm_wer = wer(all_ground_truths, kenlm_preds)
    kenlm_cer = cer(all_ground_truths, kenlm_preds)

    # --- Final Results ---
    print("\n" + "="*45)
    print(f"{'Metric':<10} | {'Raw Model':<12} | {'With KenLM':<12}")
    print("-" * 45)
    print(f"{'WER':<10} | {baseline_wer:<12.4f} | {kenlm_wer:<12.4f}")
    print(f"{'CER':<10} | {baseline_cer:<12.4f} | {kenlm_cer:<12.4f}")
    print("="*45)
    
    improvement = baseline_wer - kenlm_wer
    print(f"\nWER Improvement: {improvement:.4f}")
    
    return {"raw_wer": baseline_wer, "kenlm_wer": kenlm_wer}

In [22]:
### Optimal Checkpoint on CV ONly
results = evaluate_kenlm_manual(common_voice_test, alpha=0.5, beta=3.0)

--- Step 1: Generating Logits for 1224 samples ---


100%|██████████| 1224/1224 [01:51<00:00, 11.00it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.0538       | 0.0386      
CER        | 0.0116       | 0.0093      

WER Improvement: 0.0152


In [20]:
### Optimal Checkpoint on MASRI
results = evaluate_kenlm_manual(common_voice_test, alpha=0.5, beta=3.0)

--- Step 1: Generating Logits for 668 samples ---


100%|██████████| 668/668 [01:26<00:00,  7.68it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.2241       | 0.1989      
CER        | 0.0747       | 0.0741      

WER Improvement: 0.0252


In [21]:
### Checking the optimal checkpoint on combined
results = evaluate_kenlm_manual(common_voice_test, alpha=0.5, beta=3.0)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [03:05<00:00, 10.20it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1360       | 0.1160      
CER        | 0.0416       | 0.0401      

WER Improvement: 0.0200


In [20]:
# Masri Test
results = evaluate_kenlm_manual(common_voice_test, alpha=0.5, beta=3.0)

--- Step 1: Generating Logits for 668 samples ---


100%|██████████| 668/668 [01:25<00:00,  7.80it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.2262       | 0.1966      
CER        | 0.0729       | 0.0712      

WER Improvement: 0.0296


In [23]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.5, beta=3.0)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [03:05<00:00, 10.21it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1374       | 0.1148      
CER        | 0.0408       | 0.0386      

WER Improvement: 0.0226


In [24]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.5, beta=3.0)

--- Step 1: Generating Logits for 1224 samples ---


100%|██████████| 1224/1224 [02:16<00:00,  8.98it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.0546       | 0.0385      
CER        | 0.0117       | 0.0092      

WER Improvement: 0.0161


In [ ]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.7, beta=3.0)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [02:58<00:00, 10.60it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.7, Beta: 3.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1833       | 0.1457      
CER        | 0.0578       | 0.0552      

WER Improvement: 0.0376


Alright! The transcription can definitely be recognized from our prediction, but it is not perfect yet. Training the model a bit longer, spending more time on the data preprocessing, and especially using a language model for decoding would certainly improve the model's overall performance.

For a demonstration model on a low-resource language, the results are quite acceptable however 🤗.

#### Evaluating with beam width

In [22]:
def evaluate_kenlm_manual(dataset, alpha=0.85, beta=4.0):
    all_logits_numpy = []
    all_ground_truths = []
    greedy_preds = []

    print(f"--- Step 1: Generating Logits for {len(dataset)} samples ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            logits = model(input_features).logits
        
        all_logits_numpy.append(logits.cpu().numpy()[0])
        all_ground_truths.append(batch["sentence"].lower())
        
        pred_ids = torch.argmax(logits, dim=-1)
        greedy_preds.append(processor.batch_decode(pred_ids)[0].lower())

    # Calculate Raw Metrics
    baseline_wer = wer(all_ground_truths, greedy_preds)
    baseline_cer = cer(all_ground_truths, greedy_preds)

    # --- Step 2: Decode with fixed KenLM parameters (Default Beam 100) ---
    print(f"\n--- Step 2: Decoding with KenLM (Alpha: {alpha}, Beta: {beta}, Beam: 100) ---")
    
    decoder = build_ctcdecoder(
        labels=sorted_vocab,
        kenlm_model_path=LM_PATH,
        unigrams=unigrams, 
        alpha=alpha,
        beta=beta
    )
    
    kenlm_preds = [decoder.decode(l).lower() for l in all_logits_numpy]
    kenlm_wer = wer(all_ground_truths, kenlm_preds)
    kenlm_cer = cer(all_ground_truths, kenlm_preds)

    # --- Step 3: Beam Sensitivity Analysis ---
    # This specifically addresses your thesis question: "Does search depth matter?"
    print(f"\n--- Step 3: Beam Sensitivity Analysis (Testing Search Depth) ---")
    print(f"{'Beam Width':<12} | {'WER':<10} | {'Time (s)':<10}")
    print("-" * 35)
    
    beam_results = {}
    for bw in [32, 100, 128, 512]: # Narrow, slightly above default, and deep
        start_time = time.time()
        # We pass beam_width explicitly here
        preds = [decoder.decode(l, beam_width=bw).lower() for l in all_logits_numpy]
        bw_wer = wer(all_ground_truths, preds)
        elapsed = time.time() - start_time
        
        print(f"{bw:<12} | {bw_wer:<10.4f} | {elapsed:<10.2f}")
        beam_results[bw] = bw_wer

    # --- Final Results Report ---
    print("\n" + "="*45)
    print(f"{'Metric':<10} | {'Raw Model':<12} | {'With KenLM':<12}")
    print("-" * 45)
    print(f"{'WER':<10} | {baseline_wer:<12.4f} | {kenlm_wer:<12.4f}")
    print(f"{'CER':<10} | {baseline_cer:<12.4f} | {kenlm_cer:<12.4f}")
    print("="*45)
    
    improvement = baseline_wer - kenlm_wer
    print(f"\nBaseline vs KenLM (100 beam) Improvement: {improvement:.4f}")
    
    # Thesis-ready insight:
    deep_gain = kenlm_wer - beam_results[512]
    if deep_gain > 0.001:
        print(f"Note: Increasing beam to 512 unlocked an additional {deep_gain:.4f} WER reduction.")
    else:
        print("Note: Increasing beam width beyond 100 yielded no significant gain.")
    
    return {"raw_wer": baseline_wer, "kenlm_wer": kenlm_wer, "beam_results": beam_results}

#### Trigram Big model 

In [23]:
results = evaluate_kenlm_manual(common_voice_dev, alpha=0.5, beta=3.0)

--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:47<00:00, 11.25it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0, Beam: 100) ---

--- Step 3: Beam Sensitivity Analysis (Testing Search Depth) ---
Beam Width   | WER        | Time (s)  
-----------------------------------
32           | 0.1079     | 71.63     
100          | 0.1064     | 110.06    
128          | 0.1067     | 122.44    
512          | 0.1076     | 301.32    

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1237       | 0.1064      
CER        | 0.0358       | 0.0339      

Baseline vs KenLM (100 beam) Improvement: 0.0173
Note: Increasing beam width beyond 100 yielded no significant gain.


In [27]:
results = evaluate_kenlm_manual(common_voice_test, alpha=0.7, beta=3.0)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [02:59<00:00, 10.54it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.7, Beta: 3.0, Beam: 100) ---

--- Step 3: Beam Sensitivity Analysis (Testing Search Depth) ---
Beam Width   | WER        | Time (s)  
-----------------------------------
32           | 0.1475     | 44.04     
128          | 0.1424     | 81.87     
512          | 0.1419     | 270.34    

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1842       | 0.1429      
CER        | 0.0573       | 0.0538      

Baseline vs KenLM (100 beam) Improvement: 0.0413
Note: Increasing beam to 512 unlocked an additional 0.0010 WER reduction.


#### Normalised Test

In [26]:
import torch
import numpy as np
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer
from tqdm import tqdm

# ==========================================
# 1. Normalization Protocol for Fair Testing
# ==========================================
def normalize_maltese_text(text):
    """
    Maps low-frequency/noisy characters back to the core Maltese set.
    This ensures models with larger vocabularies are not penalized for 
    phonetic variants not present in the ground truth.
    """
    if not text:
        return ""
    
    # Define mapping for characters with <20 occurrences or foreign accents
    mapping = {
        "á": "a", 
        "é": "e", 
        "ć": "ċ",
        "`": "",   # Remove backticks/noise symbols
        "´": "",
    }
    
    text = text.lower()
    for noisy_char, clean_char in mapping.items():
        text = text.replace(noisy_char, clean_char)
        
    # Standardize whitespace
    return " ".join(text.split())

In [27]:
# ==========================================
# 2. Setup Paths & Device
# ==========================================
base_path = Path(".") 
# Update this path to the specific checkpoint you are testing (Split, Mid, or Reduced)
my_model_path = base_path / "Runs" / "SplitRunReduced" / "checkpoint-2780"
LM_PATH = str(base_path / "3_tok_red.bin")

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 3. Load Model and Processor
# ==========================================
print(f"Loading model from: {my_model_path}")
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

# Prepare Decoder Vocab
vocab = processor.tokenizer.get_vocab()
sorted_vocab = [k for k, v in sorted(vocab.items(), key=lambda item: item[1])]

# ==========================================
# 4. Evaluation Function
# ==========================================
def evaluate_kenlm_manual(dataset, alpha=0.85, beta=4.0):
    all_logits_numpy = []
    all_ground_truths = []
    greedy_preds = []

    print(f"--- Step 1: Generating Logits for {len(dataset)} samples ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            logits = model(input_features).logits
        
        all_logits_numpy.append(logits.cpu().numpy()[0])
        all_ground_truths.append(batch["sentence"]) # Lowercasing handled in normalization
        
        pred_ids = torch.argmax(logits, dim=-1)
        greedy_preds.append(processor.batch_decode(pred_ids)[0])

    # --- Step 2: Decoding with KenLM ---
    print(f"\n--- Step 2: Decoding with KenLM (Alpha: {alpha}, Beta: {beta}) ---")
    
    decoder = build_ctcdecoder(
        labels=sorted_vocab,
        kenlm_model_path=LM_PATH,
        alpha=alpha,
        beta=beta
    )
    
    # Generate raw KenLM predictions
    kenlm_preds = [decoder.decode(l) for l in all_logits_numpy]

    # --- Step 3: Apply Normalization for Fair Comparison ---
    # This step is CRITICAL for the thesis to isolate acoustic performance
    clean_ground_truths = [normalize_maltese_text(gt) for gt in all_ground_truths]
    clean_greedy_preds = [normalize_maltese_text(p) for p in greedy_preds]
    clean_kenlm_preds = [normalize_maltese_text(p) for p in kenlm_preds]

    # --- Step 4: Calculate Metrics ---
    baseline_wer = wer(clean_ground_truths, clean_greedy_preds)
    baseline_cer = cer(clean_ground_truths, clean_greedy_preds)
    
    kenlm_wer = wer(clean_ground_truths, clean_kenlm_preds)
    kenlm_cer = cer(clean_ground_truths, clean_kenlm_preds)

    # --- Final Results Output ---
    print("\n" + "="*45)
    print(f"{'Metric':<10} | {'Raw Model':<12} | {'With KenLM':<12}")
    print("-" * 45)
    print(f"{'WER':<10} | {baseline_wer:<12.4f} | {kenlm_wer:<12.4f}")
    print(f"{'CER':<10} | {baseline_cer:<12.4f} | {kenlm_cer:<12.4f}")
    print("="*45)
    
    improvement = baseline_wer - kenlm_wer
    print(f"\nNormalized WER Improvement: {improvement:.4f}")
    
    return {"raw_wer": baseline_wer, "kenlm_wer": kenlm_wer}

Loading model from: Runs/SplitRunReduced/checkpoint-2780


In [28]:
results = evaluate_kenlm_manual(common_voice_test)

--- Step 1: Generating Logits for 1892 samples ---


100%|██████████| 1892/1892 [02:59<00:00, 10.55it/s]
Unigrams not provided and cannot be automatically determined from LM file (only arpa format). Decoding accuracy might be reduced.
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?
No known unigrams provided, decoding results might be a lot worse.



--- Step 2: Decoding with KenLM (Alpha: 0.85, Beta: 4.0) ---

Metric     | Raw Model    | With KenLM  
---------------------------------------------
WER        | 0.1833       | 0.1539      
CER        | 0.0579       | 0.0582      

Normalized WER Improvement: 0.0294



## Scaling-up the training

We've shown in this blogpost how Meta's `w2v-bert-2.0` fine-tuning can give near state-of-the-art performance on low-resource languages.

To take things a step further, I've put together a set of tips and pointers given by my colleagues at Hugging Face on how to scale up training for this model. Many thanks to [Patrick](https://huggingface.co/patrickvonplaten), [Sanchit](https://huggingface.co/sanchit-gandhi) and [Pablo](https://huggingface.co/Molbap) for their valuable expertise and help 🤗

Note that Common Voice newest version ([CV16](https://huggingface.co/datasets/mozilla-foundation/common_voice_16_0)) provides many more hours of data and for may languages and thus provides fertile ground for much more efficient models in many low-resource languages.


### Datasets-related tips

CTC ASR is typically done with lower-case, un-punctuated transcriptions. This simplifies the CTC task since the model is considered as "acoustic only", meaning that it makes prediction largely based on the phonetics sounds of the audio, rather than any language modelling context of the spoken sentence.

Very low-frequency characters can significantly affect loss during learning by causing loss spikes via erroneous targets. By default, the CTC tokenizer created in this blog post would add them to the vocabulary even if their frequency is negligible compared to more frequent characters. We can treat these characters as "errors" in the dataset annotation, so that they can be removed from the vocabulary, and simply classified as `"[UNK]"` during training.

It is therefore absolutely necessary to recheck the tokenizer vocabulary and remove all low-frequency characters, in much the same way as we removed Latin characters when creating the tokenizer.

Note that the Common Voice dataset is particularly prone to such "wrong" characters, for example characters from other languages (阪).

### Training-related tips

**Average duration seen by each CTC token:** According to the advice of my colleagues, the ideal ratio of duration seen per CTC token is 10 to 35 ms. In other words, to be able to learn and predict correctly, the duration of the acoustic information a CTC token needs to see should be neither too low nor too high. In fact, it should more or less correspond to a fraction of the time it takes us humans to pronounce a phoneme.

This is something that came to light when I started refining this new W2V2-Bert model (you can find the logs of one of my experiments [here](https://wandb.ai/ylacombe/huggingface/runs/4y8pd2gq)). I quickly noticed that the loss curve of my model was initially going nicely downwards, as expected, but at some point it started to explode.

After discussing this with my colleagues, I realized that I had been using a [basic checkpoint with no architecture changes](https://huggingface.co/facebook/w2v-bert-2.0), and that each CTC token was seeing a piece of the signal for 30 to 60 ms. Adding an adapter layer was enough to reduce the signal chunk sampling to the desired duration.


**Under-training:** After showing this blog post [training run](https://huggingface.co/hf-audio/w2v-bert-2.0-mongolian-colab-CV16.0#training-results) and another [training attempt](https://wandb.ai/ylacombe/huggingface/runs/nasaux7f?workspace=user-ylacombe) on English Common Voice (about 2.5k validated hours) to my colleagues, they pointed out that the model was severely under-trained, something that could have been spotted by looking at the loss curve, which looks like it was stopped in the middle of a steep descent. This pointed out other issues as well, notably the loss curve not being smooth enough, a sign of wrong hyper-parameters settings.

Here are a few ways to solve under-training in our case:
- the warm-up rate might be too high, causing the learning rate to drop too quickly. A way to solve this would be keep the warmup ratio to 5 to 15% and scale up the number of epochs. The warm-up steps are essential to gradually bring the new language-model head weights into alignment with the pre-trained model.
- Loss curve lack of smoothness can be played around thanks to [AdamW](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html)'s [\\( \beta_2 \\)](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments.adam_beta2) which can typically set from 0.95 to 0.98 by default.




*Related posts and additional links are listed here:*
- [**Official paper**](https://huggingface.co/papers/2305.13516)
- [**Original cobebase**](https://ai.meta.com/research/publications/seamless-multilingual-expressive-and-streaming-speech-translation/)
- [**Transformers Docs**](https://huggingface.co/docs/transformers/main/en/model_doc/wav2vec2-bert)
- [**Related XLS-R blog post**](https://huggingface.co/blog/fine-tune-xlsr-wav2vec2)
- [**Related MMS blog post**](https://huggingface.co/blog/mms_adapters)


In [ ]:
import torch
import time
import pandas as pd
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer, process_words
from tqdm import tqdm

# =====================================================================
# 1. Setup Paths & Device
# =====================================================================
base_path = Path(".") 
my_model_path = base_path / "Runs" / "checkpoint-5690"
LM_PATH = str(base_path / "3_untok_red.bin")
ARPA_PATH = str(base_path / "3_untok_red.arpa") 

device = "cuda" if torch.cuda.is_available() else "cpu"

# =====================================================================
# 2. Load Model, Processor, and Vocab
# =====================================================================
print("Loading model and processor...")
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

vocab = processor.tokenizer.get_vocab()
sorted_vocab = [k for k, v in sorted(vocab.items(), key=lambda item: item[1])]

# =====================================================================
# 3. Extract Unigrams from ARPA
# =====================================================================
def get_unigram_list(arpa_path):
    unigrams = []
    with open(arpa_path, "r") as f:
        is_unigram_section = False
        for line in f:
            if "\\1-grams:" in line:
                is_unigram_section = True
                continue
            if "\\2-grams:" in line or line.startswith("\\end\\"):
                break
            if is_unigram_section:
                parts = line.strip().split()
                if len(parts) >= 2:
                    word = parts[1]
                    if word not in ["<s>", "</s>", "<unk>"]:
                        unigrams.append(word)
    return unigrams

print("Extracting unigrams...")
unigrams = get_unigram_list(ARPA_PATH)

# =====================================================================
# 4. Evaluation Function with Granular Logging
# =====================================================================
def evaluate_kenlm_manual_with_logging(dataset, alpha=0.85, beta=4.0, output_csv="asr_evaluation_big_model.csv"):
    all_logits_numpy = []
    all_ground_truths = []
    greedy_preds = []
    ids = []

    # --- Step 1: Generate Logits & Greedy Predictions ---
    print(f"\n--- Step 1: Generating Logits for {len(dataset)} samples ---")
    
    # New lists to keep track of metadata
    filenames = []
    domains = []
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        # Pull metadata straight from your loaded dataset schema
        fname = batch.get("sentence_id", f"unknown_{i}")
        domain = batch.get("sentence_domain", "unknown")
        
        filenames.append(fname)
        domains.append(domain)
        
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            logits = model(input_features).logits
        
        all_logits_numpy.append(logits.cpu().numpy()[0])
        all_ground_truths.append(batch["sentence"].lower())
        
        pred_ids = torch.argmax(logits, dim=-1)
        greedy_preds.append(processor.batch_decode(pred_ids)[0].lower())

    # --- Step 2: Decode with KenLM ---
    print(f"\n--- Step 2: Decoding with KenLM (Alpha: {alpha}, Beta: {beta}) ---")
    
    decoder = build_ctcdecoder(
        labels=sorted_vocab,
        kenlm_model_path=LM_PATH,
        unigrams=unigrams, 
        alpha=alpha,
        beta=beta
    )
    
    kenlm_preds = [decoder.decode(l).lower() for l in all_logits_numpy]

    # --- Step 3: Compute Breakdown Metrics & Compile Rows ---
    print("\n--- Step 3: Computing error alignments and saving dataset ---")
    rows = []
    
    # Added 'fname' and 'domain' to the zip statement
    for fname, domain, gt, greedy, kenlm in zip(filenames, domains, all_ground_truths, greedy_preds, kenlm_preds):
        
        if gt.strip():
            g_wer = wer(gt, greedy)
            g_cer = cer(gt, greedy)
            k_wer = wer(gt, kenlm)
            k_cer = cer(gt, kenlm)
            
            greedy_align = process_words(gt, greedy)
            kenlm_align = process_words(gt, kenlm)
            
            g_sub, g_ins, g_del = greedy_align.substitutions, greedy_align.insertions, greedy_align.deletions
            k_sub, k_ins, k_del = kenlm_align.substitutions, kenlm_align.insertions, kenlm_align.deletions
        else:
            g_wer, g_cer, k_wer, k_cer = 0.0, 0.0, 0.0, 0.0
            g_sub, g_ins, g_del = 0, 0, 0
            k_sub, k_ins, k_del = 0, 0, 0

        rows.append({
            # --- Brand New Metadata Columns ---
            "source_domain": domain,        # e.g., 'MASRI' or 'CommonVoice'
            "filename": fname,              # e.g., 'audio_sample_123.wav'
            
            "ground_truth": gt,
            "greedy_pred": greedy,
            "kenlm_pred": kenlm,
            
            # Error Rates
            "greedy_wer": g_wer,
            "greedy_cer": g_cer,
            "kenlm_wer": k_wer,
            "kenlm_cer": k_cer,
            
            # Improvement Delta
            "wer_improvement": g_wer - k_wer,
            "cer_improvement": g_cer - k_cer,
            
            # Baseline Error Breakdown
            "greedy_substitutions": g_sub, 
            "greedy_insertions": g_ins,    
            "greedy_deletions": g_del,
            
            # KenLM Error Breakdown
            "kenlm_substitutions": k_sub, 
            "kenlm_insertions": k_ins,    
            "kenlm_deletions": k_del     
        })

    # Convert to DataFrame and export
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"🎉 Analysis file saved successfully to: {output_csv}")

    # Dataset-wide global summary metrics for quick CLI reference
    global_baseline_wer = wer(all_ground_truths, greedy_preds)
    global_kenlm_wer = wer(all_ground_truths, kenlm_preds)
    
    print("\n" + "="*45)
    print(f"{'Global Metric':<15} | {'Raw Model':<12} | {'With KenLM':<12}")
    print("-" * 45)
    print(f"{'WER':<15} | {global_baseline_wer:<12.4f} | {global_kenlm_wer:<12.4f}")
    print("="*45)
    print(f"Overall Delta: {global_baseline_wer - global_kenlm_wer:.4f}\n")
    
    return df

# =====================================================================
# 5. Execution Example
# =====================================================================
if __name__ == "__main__":
    # Assuming 'eval_dataset' is your dataset object loaded from Hugging Face or custom script
    # df_results = evaluate_kenlm_manual_with_logging(eval_dataset, alpha=0.85, beta=4.0)
    pass

Loading model and processor...
Extracting unigrams...


In [20]:
import torch
import time
import pandas as pd
from pathlib import Path
from transformers import Wav2Vec2BertForCTC, Wav2Vec2BertProcessor
from pyctcdecode import build_ctcdecoder
from jiwer import wer, cer, process_words
from tqdm import tqdm
import malti.tokeniser  # Imported for Maltese detokenization

# =====================================================================
# 1. Setup Paths & Device
# =====================================================================
base_path = Path(".") 
my_model_path = base_path / "Runs" / "SplitRunReducedButMid" / "checkpoint-2780"
LM_PATH = str(base_path / "3_tok_red.bin")
ARPA_PATH = str(base_path / "3_tok_red.arpa") 

device = "cuda" if torch.cuda.is_available() else "cpu"

# Initialize Maltese Detokenizer globally
malti_tokenizer = malti.tokeniser.KMTokeniser()

# =====================================================================
# 2. Load Model, Processor, and Vocab
# =====================================================================
print("Loading model and processor...")
model = Wav2Vec2BertForCTC.from_pretrained(my_model_path).to(device)
processor = Wav2Vec2BertProcessor.from_pretrained(my_model_path)
model.eval()

vocab = processor.tokenizer.get_vocab()
sorted_vocab = [k for k, v in sorted(vocab.items(), key=lambda item: item[1])]

# =====================================================================
# 3. Extract Unigrams from ARPA
# =====================================================================
def get_unigram_list(arpa_path):
    unigrams = []
    with open(arpa_path, "r") as f:
        is_unigram_section = False
        for line in f:
            if "\\1-grams:" in line:
                is_unigram_section = True
                continue
            if "\\2-grams:" in line or line.startswith("\\end\\"):
                break
            if is_unigram_section:
                parts = line.strip().split()
                if len(parts) >= 2:
                    word = parts[1]
                    if word not in ["<s>", "</s>", "<unk>"]:
                        unigrams.append(word)
    return unigrams

print("Extracting unigrams...")
unigrams = get_unigram_list(ARPA_PATH)

# =====================================================================
# 4. Evaluation Function with Granular Logging & Detokenization
# =====================================================================
def evaluate_kenlm_manual_with_logging(dataset, alpha=0.85, beta=4.0, output_csv="asr_evaluation_small_model.csv"):
    all_logits_numpy = []
    all_ground_truths = []
    greedy_preds = []
    filenames = []
    domains = []

    # --- Step 1: Generate Logits & Greedy Predictions ---
    print(f"\n--- Step 1: Generating Logits for {len(dataset)} samples ---")
    
    for i in tqdm(range(len(dataset))):
        batch = dataset[i]
        audio = batch["audio"]["array"] 
        
        # Pull metadata
        fname = batch.get("sentence_id", f"unknown_{i}")
        domain = batch.get("sentence_domain", "unknown")
        
        filenames.append(fname)
        domains.append(domain)
        
        inputs = processor(audio, sampling_rate=16000, return_tensors="pt")
        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            logits = model(input_features).logits
        
        all_logits_numpy.append(logits.cpu().numpy()[0])
        
        # 1. Process Raw Greedy Output
        pred_ids = torch.argmax(logits, dim=-1)
        raw_greedy = processor.batch_decode(pred_ids)[0].lower()
        
        # 2. Process Raw Reference
        raw_reference = batch["sentence"].lower()
        
        # --- Detokenization Step ---
        detok_ref = malti_tokenizer.detokenise(raw_reference.split())
        detok_greedy = malti_tokenizer.detokenise(raw_greedy.split())
        
        all_ground_truths.append(detok_ref)
        greedy_preds.append(detok_greedy)

    # --- Step 2: Decode with KenLM & Detokenize ---
    print(f"\n--- Step 2: Decoding with KenLM (Alpha: {alpha}, Beta: {beta}) ---")
    
    decoder = build_ctcdecoder(
        labels=sorted_vocab,
        kenlm_model_path=LM_PATH,
        unigrams=unigrams, 
        alpha=alpha,
        beta=beta
    )
    
    kenlm_preds = []
    for l in all_logits_numpy:
        raw_kenlm = decoder.decode(l).lower()
        # Detokenize the KenLM output string
        detok_kenlm = malti_tokenizer.detokenise(raw_kenlm.split())
        kenlm_preds.append(detok_kenlm)

    # --- Step 3: Compute Breakdown Metrics & Compile Rows ---
    print("\n--- Step 3: Computing error alignments and saving dataset ---")
    rows = []
    
    for fname, domain, gt, greedy, kenlm in zip(filenames, domains, all_ground_truths, greedy_preds, kenlm_preds):
        
        if gt.strip():
            g_wer = wer(gt, greedy)
            g_cer = cer(gt, greedy)
            k_wer = wer(gt, kenlm)
            k_cer = cer(gt, kenlm)
            
            greedy_align = process_words(gt, greedy)
            kenlm_align = process_words(gt, kenlm)
            
            g_sub, g_ins, g_del = greedy_align.substitutions, greedy_align.insertions, greedy_align.deletions
            k_sub, k_ins, k_del = kenlm_align.substitutions, kenlm_align.insertions, kenlm_align.deletions
        else:
            g_wer, g_cer, k_wer, k_cer = 0.0, 0.0, 0.0, 0.0
            g_sub, g_ins, g_del = 0, 0, 0
            k_sub, k_ins, k_del = 0, 0, 0

        rows.append({
            "source_domain": domain,        
            "filename": fname,              
            
            "ground_truth": gt,
            "greedy_pred": greedy,
            "kenlm_pred": kenlm,
            
            # Error Rates (Calculated on Fair Detokenized text)
            "greedy_wer": g_wer,
            "greedy_cer": g_cer,
            "kenlm_wer": k_wer,
            "kenlm_cer": k_cer,
            
            # Improvement Delta
            "wer_improvement": g_wer - k_wer,
            "cer_improvement": g_cer - k_cer,
            
            # Baseline Error Breakdown
            "greedy_substitutions": g_sub, 
            "greedy_insertions": g_ins,    
            "greedy_deletions": g_del,
            
            # KenLM Error Breakdown
            "kenlm_substitutions": k_sub, 
            "kenlm_insertions": k_ins,    
            "kenlm_deletions": k_del     
        })

    # Convert to DataFrame and export
    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f"🎉 Analysis file saved successfully to: {output_csv}")

    # Dataset-wide global summary metrics (Fair Evaluation)
    global_baseline_wer = wer(all_ground_truths, greedy_preds)
    global_baseline_cer = cer(all_ground_truths, greedy_preds)
    global_kenlm_wer = wer(all_ground_truths, kenlm_preds)
    global_kenlm_cer = cer(all_ground_truths, kenlm_preds)
    
    print("\n" + "="*50)
    print(f"{'Global Metric (Detok)':<20} | {'Raw Model':<12} | {'With KenLM':<12}")
    print("-" * 50)
    print(f"{'WER':<20} | {global_baseline_wer:<12.4f} | {global_kenlm_wer:<12.4f}")
    print(f"{'CER':<20} | {global_baseline_cer:<12.4f} | {global_kenlm_cer:<12.4f}")
    print("="*50)
    print(f"Overall WER Delta: {global_baseline_wer - global_kenlm_wer:.4f}\n")
    
    return df

# =====================================================================
# 5. Execution Example
# =====================================================================
if __name__ == "__main__":
    # Assuming 'eval_dataset' is your dataset object loaded from Hugging Face or custom script
    # df_results = evaluate_kenlm_manual_with_logging(eval_dataset, alpha=0.85, beta=4.0)
    pass

Loading model and processor...
Extracting unigrams...


In [21]:
results = evaluate_kenlm_manual_with_logging(common_voice_dev, alpha=0.7, beta=3.0)


--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [03:39<00:00,  8.60it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.7, Beta: 3.0) ---

--- Step 3: Computing error alignments and saving dataset ---
🎉 Analysis file saved successfully to: asr_evaluation_small_model.csv

Global Metric (Detok) | Raw Model    | With KenLM  
--------------------------------------------------
WER                  | 0.1989       | 0.1526      
CER                  | 0.0516       | 0.0478      
Overall WER Delta: 0.0463



In [19]:
results = evaluate_kenlm_manual_with_logging(common_voice_dev, alpha=0.5, beta=3.0)


--- Step 1: Generating Logits for 1883 samples ---


100%|██████████| 1883/1883 [02:59<00:00, 10.48it/s]
Found entries of length > 1 in alphabet. This is unusual unless style is BPE, but the alphabet was not recognized as BPE type. Is this correct?



--- Step 2: Decoding with KenLM (Alpha: 0.5, Beta: 3.0) ---

--- Step 3: Computing error alignments and saving dataset ---
🎉 Analysis file saved successfully to: asr_evaluation_big_model.csv

Global Metric   | Raw Model    | With KenLM  
---------------------------------------------
WER             | 0.1237       | 0.1064      
Overall Delta: 0.0173

